In [ ]:
import os
import cv2
import torch
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from ultralytics import YOLO
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple, Set
from collections import defaultdict
import json
import math

# ================= 环境设置 =================
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["XFORMERS_DISABLED"] = "1"

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# ================= 模型加载 =================

# 1. GazeLLE
torch.hub.set_dir(r'/Users/zehao/Desktop/HAI/MultiParty/code/gazelle-main')
gazelle_model, gazelle_transform = torch.hub.load(
    'fkryan/gazelle', 'gazelle_dinov2_vitb14_inout'
)
gazelle_model.eval()
gazelle_model.to(device)

# 2. YOLO Face Detector
face_detector = YOLO('yolov8n-face.pt')


# ================================================================
#                    MODULE A1: Person Detection + Tracking
# ================================================================

@dataclass
class TrackedPerson:
    """一个被跟踪的人物"""
    person_id: int
    bbox: List[float]          # [x1, y1, x2, y2] 像素坐标
    bbox_norm: List[float]     # [x1, y1, x2, y2] 归一化坐标
    confidence: float
    last_seen_frame: int
    # 以下在A2中填充
    gaze_target_pos: Optional[Tuple[float, float]] = None   # 注视点归一化坐标
    gaze_target_person: Optional[int] = None                 # 注视目标的person_id
    gaze_target_type: str = "unknown"                        # "person" / "object" / "out_of_frame" / "unknown"
    gaze_heatmap: Optional[np.ndarray] = None
    inout_score: float = 0.0
    head_direction: Optional[Tuple[float, float]] = None     # 头部朝向向量 (dx, dy)
    gaze_confidence: float = 0.0


class PersonTracker:
    """
    基于IoU的简易多人跟踪器
    
    核心逻辑：
    - 每帧YOLO检测人脸 → 与上一帧的tracked persons做IoU匹配
    - IoU > 阈值 → 同一个人，保持ID
    - 无法匹配 → 新人，分配新ID
    - 连续多帧未出现 → 移除
    """
    
    def __init__(self, iou_threshold=0.3, max_disappear=30, max_persons=10):
        self.iou_threshold = iou_threshold
        self.max_disappear = max_disappear   # 最多消失多少帧后移除
        self.max_persons = max_persons
        self.next_id = 1
        self.tracked: Dict[int, TrackedPerson] = {}  # person_id → TrackedPerson
        self.current_frame = 0
    
    @staticmethod
    def compute_iou(box1, box2):
        """计算两个bbox的IoU, 输入为 [x1, y1, x2, y2] 像素坐标"""
        x1 = max(box1[0], box2[0])
        y1 = max(box1[1], box2[1])
        x2 = min(box1[2], box2[2])
        y2 = min(box1[3], box2[3])
        
        inter = max(0, x2 - x1) * max(0, y2 - y1)
        area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
        area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
        union = area1 + area2 - inter
        
        return inter / union if union > 0 else 0
    
    def update(self, detections: List[List[float]], confidences: List[float],
               frame_idx: int, img_width: int, img_height: int) -> Dict[int, TrackedPerson]:
        """
        每帧调用：输入当前帧检测结果，返回更新后的tracked persons
        
        Args:
            detections: [[x1,y1,x2,y2], ...] 像素坐标
            confidences: [conf1, conf2, ...]
            frame_idx: 当前帧号
            img_width, img_height: 图像尺寸
        
        Returns:
            当前帧活跃的 {person_id: TrackedPerson}
        """
        self.current_frame = frame_idx
        
        # 1. 获取当前所有活跃的tracked persons
        active_ids = list(self.tracked.keys())
        active_boxes = [self.tracked[pid].bbox for pid in active_ids]
        
        # 2. 计算IoU矩阵
        matched_det = set()
        matched_track = set()
        
        if len(active_ids) > 0 and len(detections) > 0:
            iou_matrix = np.zeros((len(active_ids), len(detections)))
            for i, track_box in enumerate(active_boxes):
                for j, det_box in enumerate(detections):
                    iou_matrix[i, j] = self.compute_iou(track_box, det_box)
            
            # 3. 贪心匹配：每次选IoU最大的配对
            while True:
                if iou_matrix.size == 0:
                    break
                max_iou = np.max(iou_matrix)
                if max_iou < self.iou_threshold:
                    break
                
                i, j = np.unravel_index(np.argmax(iou_matrix), iou_matrix.shape)
                
                # 匹配成功：更新已有track
                pid = active_ids[i]
                self.tracked[pid].bbox = detections[j]
                self.tracked[pid].bbox_norm = [
                    detections[j][0] / img_width,
                    detections[j][1] / img_height,
                    detections[j][2] / img_width,
                    detections[j][3] / img_height,
                ]
                self.tracked[pid].confidence = confidences[j]
                self.tracked[pid].last_seen_frame = frame_idx
                # 重置gaze信息（每帧重新计算）
                self._reset_gaze(pid)
                
                matched_det.add(j)
                matched_track.add(i)
                
                # 将已匹配的行列设为-1
                iou_matrix[i, :] = -1
                iou_matrix[:, j] = -1
        
        # 4. 未匹配的检测 → 新人物
        for j in range(len(detections)):
            if j not in matched_det:
                if len(self.tracked) < self.max_persons:
                    pid = self.next_id
                    self.next_id += 1
                    self.tracked[pid] = TrackedPerson(
                        person_id=pid,
                        bbox=detections[j],
                        bbox_norm=[
                            detections[j][0] / img_width,
                            detections[j][1] / img_height,
                            detections[j][2] / img_width,
                            detections[j][3] / img_height,
                        ],
                        confidence=confidences[j],
                        last_seen_frame=frame_idx
                    )
        
        # 5. 清理长时间消失的人物
        to_remove = []
        for pid, person in self.tracked.items():
            if frame_idx - person.last_seen_frame > self.max_disappear:
                to_remove.append(pid)
        for pid in to_remove:
            del self.tracked[pid]
        
        # 6. 返回当前帧活跃的人物
        active = {
            pid: person for pid, person in self.tracked.items()
            if person.last_seen_frame == frame_idx
        }
        return active
    
    def _reset_gaze(self, pid):
        """重置某人的gaze信息（每帧重新计算）"""
        p = self.tracked[pid]
        p.gaze_target_pos = None
        p.gaze_target_person = None
        p.gaze_target_type = "unknown"
        p.gaze_heatmap = None
        p.inout_score = 0.0
        p.head_direction = None
        p.gaze_confidence = 0.0


# ================================================================
#                    MODULE A2: Gaze Detection with Person IDs
# ================================================================

class GazeDetector:
    """
    使用GazeLLE检测注视目标，并将注视点映射到Person ID
    """
    
    def __init__(self, model, transform, device, 
                 inout_threshold=0.5, 
                 gaze_to_person_radius=0.15):
        """
        Args:
            gaze_to_person_radius: 注视点距离某人bbox中心多近时
                                   判定为"在看那个人"(归一化坐标)
        """
        self.model = model
        self.transform = transform
        self.device = device
        self.inout_threshold = inout_threshold
        self.gaze_to_person_radius = gaze_to_person_radius
    
    def detect(self, pil_image: Image.Image, 
               active_persons: Dict[int, TrackedPerson]) -> Dict[int, TrackedPerson]:
        """
        对当前帧所有活跃人物进行gaze检测
        
        Args:
            pil_image: 当前帧PIL图像
            active_persons: {person_id: TrackedPerson} 当前帧活跃人物
        
        Returns:
            更新了gaze信息的active_persons
        """
        if len(active_persons) == 0:
            return active_persons
        
        # 1. 准备GazeLLE输入
        img_tensor = self.transform(pil_image).unsqueeze(0).to(self.device)
        
        pid_list = list(active_persons.keys())
        norm_bboxes = []
        for pid in pid_list:
            person = active_persons[pid]
            norm_bboxes.append(person.bbox_norm)
        
        norm_bboxes_np = [np.array(norm_bboxes)]
        
        model_input = {
            "images": img_tensor,
            "bboxes": norm_bboxes_np
        }
        
        # 2. GazeLLE推理
        with torch.no_grad():
            output = self.model(model_input)
        
        heatmaps = output['heatmap'][0]  # [N, H_hm, W_hm]
        inout_scores = output['inout'][0] if output['inout'] is not None else None
        
        # 3. 解析每个人的gaze结果
        for i, pid in enumerate(pid_list):
            person = active_persons[pid]
            
            heatmap = heatmaps[i].detach().cpu().numpy()
            inout = inout_scores[i].item() if inout_scores is not None else 1.0
            
            person.inout_score = inout
            person.gaze_heatmap = heatmap
            
            if inout > self.inout_threshold:
                # 找到heatmap最大值位置 → 注视点
                max_idx = np.unravel_index(np.argmax(heatmap), heatmap.shape)
                gaze_y = max_idx[0] / heatmap.shape[0]  # 归一化 [0,1]
                gaze_x = max_idx[1] / heatmap.shape[1]  # 归一化 [0,1]
                
                person.gaze_target_pos = (gaze_x, gaze_y)
                person.gaze_confidence = float(np.max(heatmap))
                
                # 计算头部朝向（从人脸中心到注视点的方向向量）
                face_cx = (person.bbox_norm[0] + person.bbox_norm[2]) / 2
                face_cy = (person.bbox_norm[1] + person.bbox_norm[3]) / 2
                dx = gaze_x - face_cx
                dy = gaze_y - face_cy
                norm = math.sqrt(dx**2 + dy**2) + 1e-8
                person.head_direction = (dx / norm, dy / norm)
                
                # 4. 判断注视目标是哪个人
                person.gaze_target_person, person.gaze_target_type = \
                    self._map_gaze_to_person(
                        gaze_x, gaze_y, pid, active_persons
                    )
            else:
                person.gaze_target_type = "out_of_frame"
                person.gaze_confidence = inout
        
        return active_persons
    
    def _map_gaze_to_person(self, gaze_x: float, gaze_y: float,
                            self_pid: int,
                            active_persons: Dict[int, TrackedPerson]
                            ) -> Tuple[Optional[int], str]:
        """
        将注视点映射到最近的人物
        
        逻辑：
        1. 检查注视点是否落在某人的bbox内 → 直接判定
        2. 如果不在任何人bbox内，检查距离最近的人是否在阈值内
        3. 否则判定为看物体/场景
        """
        best_pid = None
        best_dist = float('inf')
        
        for pid, person in active_persons.items():
            if pid == self_pid:
                continue  # 不考虑自己
            
            bx1, by1, bx2, by2 = person.bbox_norm
            
            # 扩展bbox (人可能看的是对方的身体而非仅脸部)
            expand = 0.05
            bx1_e = max(0, bx1 - expand)
            by1_e = max(0, by1 - expand)
            bx2_e = min(1, bx2 + expand)
            by2_e = min(1, by2 + expand)
            
            # 检查注视点是否在扩展bbox内
            if bx1_e <= gaze_x <= bx2_e and by1_e <= gaze_y <= by2_e:
                # 在bbox内，计算到中心距离作为排序依据
                cx = (bx1 + bx2) / 2
                cy = (by1 + by2) / 2
                dist = math.sqrt((gaze_x - cx)**2 + (gaze_y - cy)**2)
                if dist < best_dist:
                    best_dist = dist
                    best_pid = pid
            else:
                # 不在bbox内，计算到bbox中心的距离
                cx = (bx1 + bx2) / 2
                cy = (by1 + by2) / 2
                dist = math.sqrt((gaze_x - cx)**2 + (gaze_y - cy)**2)
                if dist < self.gaze_to_person_radius and dist < best_dist:
                    best_dist = dist
                    best_pid = pid
        
        if best_pid is not None:
            return best_pid, "person"
        else:
            return None, "object"


# ================================================================
#               MODULE B: Social Mind Graph Construction
# ================================================================

@dataclass
class GraphNode:
    """Social Mind Graph的节点"""
    person_id: int
    observable_state: dict  # 可观察状态
    confidence: dict        # 各信号的置信度

@dataclass  
class GraphEdge:
    """Social Mind Graph的边（可观察关系）"""
    source_id: int
    target_id: int
    edge_type: str          # "GAZES_AT", "PROXIMATE"
    confidence: float
    attributes: dict = field(default_factory=dict)

@dataclass
class DerivedRelation:
    """Social Mind Graph的派生关系"""
    relation_type: str      # "CAN_OBSERVE", "MUTUAL_GAZE", etc.
    participants: List[int] # 涉及的person_ids
    value: bool             # True / False
    confidence: float
    reasoning: str          # 推导理由（可解释性）


class SocialMindGraph:
    """
    Social Mind Graph：形式化的社交场景表示
    
    从Module A的感知信号构建：
    - Nodes: 每个人 + 可观察状态
    - Observable Edges: GAZES_AT, PROXIMATE
    - Derived Relations: CAN_OBSERVE, MUTUAL_GAZE, 
                         ENGAGED_WITH, UNRECIPROCATED
    - Visual Access Matrix: 谁能看到谁
    """
    
    def __init__(self, fov_angle_deg=150):
        """
        Args:
            fov_angle_deg: 估计的人类水平视野角度
                          (实际约180°，但有效注意约120-150°)
        """
        self.fov_angle = math.radians(fov_angle_deg)
        
        # Graph组件
        self.nodes: Dict[int, GraphNode] = {}
        self.edges: List[GraphEdge] = []
        self.derived_relations: List[DerivedRelation] = []
        self.visual_access_matrix: Dict[Tuple[int,int], dict] = {}
        
        # 快速查询索引
        self.gazes_at: Dict[int, Optional[int]] = {}    # pid → 注视目标pid
        self.gazed_by: Dict[int, List[int]] = defaultdict(list)  # pid → 被谁注视
    
    def build(self, active_persons: Dict[int, TrackedPerson],
              img_width: int, img_height: int):
        """
        从Module A输出构建完整的Social Mind Graph
        
        Args:
            active_persons: Module A输出的当前帧活跃人物
            img_width, img_height: 图像尺寸
        """
        # 清空上一帧
        self._clear()
        
        if len(active_persons) < 2:
            # 少于2人无社交关系
            for pid, person in active_persons.items():
                self._build_node(person)
            return
        
        # Step 1: 构建节点
        for pid, person in active_persons.items():
            self._build_node(person)
        
        # Step 2: 构建可观察边 (GAZES_AT, PROXIMATE)
        self._build_observable_edges(active_persons, img_width, img_height)
        
        # Step 3: 计算Visual Access Matrix
        self._compute_visual_access(active_persons)
        
        # Step 4: 推导派生关系
        self._derive_relations(active_persons)
    
    def _clear(self):
        self.nodes.clear()
        self.edges.clear()
        self.derived_relations.clear()
        self.visual_access_matrix.clear()
        self.gazes_at.clear()
        self.gazed_by.clear()
    
    # ============ Step 1: Node Construction ============
    
    def _build_node(self, person: TrackedPerson):
        node = GraphNode(
            person_id=person.person_id,
            observable_state={
                "position": {
                    "bbox_norm": person.bbox_norm,
                    "center": (
                        (person.bbox_norm[0] + person.bbox_norm[2]) / 2,
                        (person.bbox_norm[1] + person.bbox_norm[3]) / 2
                    )
                },
                "gaze": {
                    "target_person": person.gaze_target_person,
                    "target_position": person.gaze_target_pos,
                    "target_type": person.gaze_target_type,
                    "inout_score": person.inout_score
                },
                "head_direction": person.head_direction,
            },
            confidence={
                "detection": person.confidence,
                "gaze": person.gaze_confidence
            }
        )
        self.nodes[person.person_id] = node
    
    # ============ Step 2: Observable Edges ============
    
    def _build_observable_edges(self, active_persons: Dict[int, TrackedPerson],
                                 img_width: int, img_height: int):
        pids = list(active_persons.keys())
        
        for pid in pids:
            person = active_persons[pid]
            
            # --- GAZES_AT edges ---
            if person.gaze_target_person is not None:
                target_pid = person.gaze_target_person
                edge = GraphEdge(
                    source_id=pid,
                    target_id=target_pid,
                    edge_type="GAZES_AT",
                    confidence=person.gaze_confidence,
                    attributes={
                        "gaze_point": person.gaze_target_pos,
                        "inout_score": person.inout_score
                    }
                )
                self.edges.append(edge)
                self.gazes_at[pid] = target_pid
                self.gazed_by[target_pid].append(pid)
            else:
                self.gazes_at[pid] = None
        
        # --- PROXIMATE edges ---
        for i in range(len(pids)):
            for j in range(i+1, len(pids)):
                pi = active_persons[pids[i]]
                pj = active_persons[pids[j]]
                
                # 计算两人bbox中心距离（归一化坐标）
                ci = ((pi.bbox_norm[0]+pi.bbox_norm[2])/2,
                      (pi.bbox_norm[1]+pi.bbox_norm[3])/2)
                cj = ((pj.bbox_norm[0]+pj.bbox_norm[2])/2,
                      (pj.bbox_norm[1]+pj.bbox_norm[3])/2)
                
                dist = math.sqrt((ci[0]-cj[0])**2 + (ci[1]-cj[1])**2)
                
                # 判断亲密度等级（基于归一化距离的近似）
                if dist < 0.15:
                    level = "intimate"
                elif dist < 0.30:
                    level = "personal"
                elif dist < 0.55:
                    level = "social"
                else:
                    level = "public"
                
                edge = GraphEdge(
                    source_id=pids[i],
                    target_id=pids[j],
                    edge_type="PROXIMATE",
                    confidence=min(pi.confidence, pj.confidence),
                    attributes={
                        "distance_norm": dist,
                        "level": level
                    }
                )
                self.edges.append(edge)
    
    # ============ Step 3: Visual Access Matrix ============
    
    def _compute_visual_access(self, active_persons: Dict[int, TrackedPerson]):
        """
        计算Visual Access Matrix: 谁能看到谁
        
        基于head_direction估计FOV锥，判断其他人是否在FOV内
        
        注意：没有body orientation时，用head_direction近似
        """
        pids = list(active_persons.keys())
        
        for pid_i in pids:
            pi = active_persons[pid_i]
            ci = ((pi.bbox_norm[0]+pi.bbox_norm[2])/2,
                  (pi.bbox_norm[1]+pi.bbox_norm[3])/2)
            
            for pid_j in pids:
                if pid_i == pid_j:
                    continue
                
                pj = active_persons[pid_j]
                cj = ((pj.bbox_norm[0]+pj.bbox_norm[2])/2,
                      (pj.bbox_norm[1]+pj.bbox_norm[3])/2)
                
                can_observe, conf, reason = self._check_visual_access(
                    pi, ci, pj, cj
                )
                
                self.visual_access_matrix[(pid_i, pid_j)] = {
                    "can_observe": can_observe,
                    "confidence": conf,
                    "reasoning": reason
                }
    
    def _check_visual_access(self, person_i: TrackedPerson, 
                              center_i: Tuple[float,float],
                              person_j: TrackedPerson,
                              center_j: Tuple[float,float]
                              ) -> Tuple[bool, float, str]:
        """
        检查person_i是否能观察到person_j
        
        Returns: (can_observe, confidence, reasoning)
        """
        # 如果没有head_direction信息（gaze检测失败），保守估计
        if person_i.head_direction is None:
            return True, 0.3, "Head direction unknown, assuming possible visibility"
        
        hd_x, hd_y = person_i.head_direction
        
        # 计算从i到j的方向向量
        dir_x = center_j[0] - center_i[0]
        dir_y = center_j[1] - center_i[1]
        dir_norm = math.sqrt(dir_x**2 + dir_y**2) + 1e-8
        dir_x /= dir_norm
        dir_y /= dir_norm
        
        # 计算head_direction和i→j方向的夹角
        cos_angle = hd_x * dir_x + hd_y * dir_y
        cos_angle = max(-1, min(1, cos_angle))  # clamp
        angle = math.acos(cos_angle)
        
        # 判断是否在FOV内
        half_fov = self.fov_angle / 2
        
        if angle <= half_fov * 0.5:
            # 在FOV中心区域 → 高置信度
            return True, 0.9, f"P{person_j.person_id} is in central FOV of P{person_i.person_id} (angle={math.degrees(angle):.0f}°)"
        elif angle <= half_fov:
            # 在FOV边缘区域 → 中等置信度
            return True, 0.6, f"P{person_j.person_id} is in peripheral FOV of P{person_i.person_id} (angle={math.degrees(angle):.0f}°)"
        else:
            # 在FOV外 → 不可见
            return False, 0.8, f"P{person_j.person_id} is outside FOV of P{person_i.person_id} (angle={math.degrees(angle):.0f}°)"
    
    # ============ Step 4: Derived Relations ============
    
    def _derive_relations(self, active_persons: Dict[int, TrackedPerson]):
        pids = list(active_persons.keys())
        
        for i in range(len(pids)):
            for j in range(len(pids)):
                if i == j:
                    continue
                pid_i = pids[i]
                pid_j = pids[j]
                
                # --- CAN_OBSERVE ---
                va = self.visual_access_matrix.get((pid_i, pid_j))
                if va:
                    self.derived_relations.append(DerivedRelation(
                        relation_type="CAN_OBSERVE",
                        participants=[pid_i, pid_j],
                        value=va["can_observe"],
                        confidence=va["confidence"],
                        reasoning=va["reasoning"]
                    ))
        
        # --- MUTUAL_GAZE ---
        for i in range(len(pids)):
            for j in range(i+1, len(pids)):
                pid_i = pids[i]
                pid_j = pids[j]
                
                i_looks_j = self.gazes_at.get(pid_i) == pid_j
                j_looks_i = self.gazes_at.get(pid_j) == pid_i
                
                if i_looks_j and j_looks_i:
                    conf = min(
                        active_persons[pid_i].gaze_confidence,
                        active_persons[pid_j].gaze_confidence
                    )
                    self.derived_relations.append(DerivedRelation(
                        relation_type="MUTUAL_GAZE",
                        participants=[pid_i, pid_j],
                        value=True,
                        confidence=conf,
                        reasoning=f"P{pid_i} gazes at P{pid_j} AND P{pid_j} gazes at P{pid_i}"
                    ))
        
        # --- UNRECIPROCATED_ENGAGEMENT ---
        for pid_i in pids:
            target = self.gazes_at.get(pid_i)
            if target is not None:
                # pid_i在看target，但target没有看pid_i
                if self.gazes_at.get(target) != pid_i:
                    conf = active_persons[pid_i].gaze_confidence
                    self.derived_relations.append(DerivedRelation(
                        relation_type="UNRECIPROCATED_ENGAGEMENT",
                        participants=[pid_i, target],
                        value=True,
                        confidence=conf,
                        reasoning=f"P{pid_i} gazes at P{target}, but P{target} does not gaze back at P{pid_i}"
                    ))
        
        # --- ATTENDED_BY (被多人注视 → 可能是焦点人物) ---
        for pid in pids:
            watchers = self.gazed_by.get(pid, [])
            if len(watchers) >= 2:
                self.derived_relations.append(DerivedRelation(
                    relation_type="ATTENTION_CENTER",
                    participants=[pid] + watchers,
                    value=True,
                    confidence=0.7,
                    reasoning=f"P{pid} is gazed at by {len(watchers)} people: {['P'+str(w) for w in watchers]}"
                ))
        
        # --- EXCLUDED_FROM (在场但无人注视且无人与其互动) ---
        for pid in pids:
            watchers = self.gazed_by.get(pid, [])
            target = self.gazes_at.get(pid)
            
            if len(watchers) == 0 and (target is None or 
                                        self.gazes_at.get(target) != pid):
                # 没人看ta，ta看的人也没有回看
                if len(pids) > 2:  # 至少3人时才有意义
                    self.derived_relations.append(DerivedRelation(
                        relation_type="POSSIBLY_EXCLUDED",
                        participants=[pid],
                        value=True,
                        confidence=0.5,
                        reasoning=f"P{pid} is not gazed at by anyone and has no reciprocated engagement"
                    ))
    
    # ============ 输出与可视化 ============
    
    def to_dict(self) -> dict:
        """将整个图导出为字典（便于序列化/传给Module C）"""
        return {
            "nodes": {
                pid: {
                    "person_id": node.person_id,
                    "observable_state": node.observable_state,
                    "confidence": node.confidence
                }
                for pid, node in self.nodes.items()
            },
            "observable_edges": [
                {
                    "source": e.source_id,
                    "target": e.target_id,
                    "type": e.edge_type,
                    "confidence": e.confidence,
                    "attributes": e.attributes
                }
                for e in self.edges
            ],
            "visual_access_matrix": {
                f"P{k[0]}->P{k[1]}": v
                for k, v in self.visual_access_matrix.items()
            },
            "derived_relations": [
                {
                    "type": r.relation_type,
                    "participants": [f"P{p}" for p in r.participants],
                    "value": r.value,
                    "confidence": r.confidence,
                    "reasoning": r.reasoning
                }
                for r in self.derived_relations
            ],
            "summary": self._generate_summary()
        }
    
    def _generate_summary(self) -> dict:
        """生成图的摘要信息"""
        gaze_pairs = []
        for pid, target in self.gazes_at.items():
            if target is not None:
                gaze_pairs.append(f"P{pid}→P{target}")
        
        mutual = [
            r for r in self.derived_relations 
            if r.relation_type == "MUTUAL_GAZE"
        ]
        unrecip = [
            r for r in self.derived_relations 
            if r.relation_type == "UNRECIPROCATED_ENGAGEMENT"
        ]
        
        return {
            "num_persons": len(self.nodes),
            "gaze_interactions": gaze_pairs,
            "mutual_gaze_pairs": [
                f"P{r.participants[0]}↔P{r.participants[1]}" for r in mutual
            ],
            "unreciprocated": [
                f"P{r.participants[0]}→P{r.participants[1]}(unrecip)" for r in unrecip
            ],
        }
    
    def to_natural_language(self) -> str:
        """
        将图结构转化为自然语言描述
        这是传给Module C / Module D 的文本接口
        """
        if len(self.nodes) == 0:
            return "No persons detected in the scene."
        
        lines = []
        lines.append(f"=== Social Mind Graph ({len(self.nodes)} persons) ===\n")
        
        # 1. 人物位置
        lines.append("[Persons & Positions]")
        for pid, node in sorted(self.nodes.items()):
            pos = node.observable_state["position"]["center"]
            lines.append(f"  Person P{pid}: center=({pos[0]:.2f}, {pos[1]:.2f})")
        
        # 2. 注视关系
        lines.append("\n[Gaze Relationships]")
        for pid in sorted(self.gazes_at.keys()):
            target = self.gazes_at[pid]
            person = self.nodes[pid]
            if target is not None:
                conf = person.confidence.get("gaze", 0)
                lines.append(f"  P{pid} GAZES_AT P{target} (conf={conf:.2f})")
            else:
                gaze_info = person.observable_state["gaze"]
                lines.append(f"  P{pid} gazes at {gaze_info['target_type']} (not a person)")
        
        # 3. Visual Access
        lines.append("\n[Visual Access Matrix]")
        pids = sorted(self.nodes.keys())
        for pid_i in pids:
            can_see = []
            cannot_see = []
            for pid_j in pids:
                if pid_i == pid_j:
                    continue
                va = self.visual_access_matrix.get((pid_i, pid_j))
                if va and va["can_observe"]:
                    can_see.append(f"P{pid_j}")
                elif va and not va["can_observe"]:
                    cannot_see.append(f"P{pid_j}")
            
            if can_see:
                lines.append(f"  P{pid_i} CAN observe: {', '.join(can_see)}")
            if cannot_see:
                lines.append(f"  P{pid_i} CANNOT observe: {', '.join(cannot_see)}")
        
        # 4. 派生关系
        lines.append("\n[Derived Social Relations]")
        for r in self.derived_relations:
            participants_str = ", ".join([f"P{p}" for p in r.participants])
            status = "✓" if r.value else "✗"
            lines.append(f"  {status} {r.relation_type}({participants_str}) "
                        f"[conf={r.confidence:.2f}]")
            lines.append(f"    Reasoning: {r.reasoning}")
        
        return "\n".join(lines)


# ================================================================
#               可视化函数（集成Person ID显示）
# ================================================================

def visualize_with_ids(pil_image: Image.Image, 
                       active_persons: Dict[int, TrackedPerson],
                       graph: SocialMindGraph,
                       inout_thresh: float = 0.5) -> Image.Image:
    """
    增强版可视化：显示Person ID + 注视线 + 关系标注
    """
    colors_map = {
        1: 'lime', 2: 'tomato', 3: 'cyan', 
        4: 'fuchsia', 5: 'yellow', 6: 'orange',
        7: 'white', 8: 'deepskyblue', 9: 'chartreuse', 10: 'hotpink'
    }
    
    overlay = pil_image.copy().convert("RGBA")
    draw = ImageDraw.Draw(overlay)
    width, height = pil_image.size
    
    try:
        font_large = ImageFont.truetype("arial.ttf", int(min(width, height) * 0.035))
        font_small = ImageFont.truetype("arial.ttf", int(min(width, height) * 0.02))
    except:
        font_large = ImageFont.load_default()
        font_small = ImageFont.load_default()
    
    for pid, person in active_persons.items():
        color = colors_map.get(pid % 10 + 1, 'white')
        x1, y1, x2, y2 = person.bbox_norm
        px1, py1 = x1 * width, y1 * height
        px2, py2 = x2 * width, y2 * height
        
        # 1. 画人脸框
        draw.rectangle([px1, py1, px2, py2], outline=color, width=3)
        
        # 2. 画Person ID标签
        label = f"P{pid}"
        draw.text((px1, max(0, py1 - 25)), label, fill=color, font=font_large)
        
        # 3. 画注视信息
        if person.inout_score > inout_thresh and person.gaze_target_pos:
            gx = person.gaze_target_pos[0] * width
            gy = person.gaze_target_pos[1] * height
            cx = (px1 + px2) / 2
            cy = (py1 + py2) / 2
            
            # 注视线
            draw.line([(cx, cy), (gx, gy)], fill=color, width=2)
            # 注视点
            draw.ellipse([(gx-6, gy-6), (gx+6, gy+6)], fill=color)
            
            # 注视目标标注
            if person.gaze_target_person is not None:
                target_label = f"→P{person.gaze_target_person}"
            else:
                target_label = f"→{person.gaze_target_type}"
            draw.text((px2 + 3, py1), target_label, fill=color, font=font_small)
    
    # 4. 画MUTUAL_GAZE高亮
    for rel in graph.derived_relations:
        if rel.relation_type == "MUTUAL_GAZE" and rel.value:
            p1, p2 = rel.participants
            if p1 in active_persons and p2 in active_persons:
                c1 = ((active_persons[p1].bbox_norm[0]+active_persons[p1].bbox_norm[2])/2 * width,
                       (active_persons[p1].bbox_norm[1]+active_persons[p1].bbox_norm[3])/2 * height)
                c2 = ((active_persons[p2].bbox_norm[0]+active_persons[p2].bbox_norm[2])/2 * width,
                       (active_persons[p2].bbox_norm[1]+active_persons[p2].bbox_norm[3])/2 * height)
                draw.line([c1, c2], fill='gold', width=4)
                mid = ((c1[0]+c2[0])/2, (c1[1]+c2[1])/2)
                draw.text(mid, "MUTUAL", fill='gold', font=font_small)
    
    return overlay.convert("RGB")


# ================================================================
#               主处理流程
# ================================================================

def process_video_full_pipeline(video_path: str, output_path: str,
                                 json_output_path: str = None,
                                 max_persons: int = 5,
                                 inout_thresh: float = 0.5,
                                 yolo_conf: float = 0.7):
    """
    完整Pipeline: Module A1 + A2 + Module B
    
    Args:
        video_path: 输入视频路径
        output_path: 输出视频路径
        json_output_path: 输出每帧图结构的JSON路径
        max_persons: 最大跟踪人数
        inout_thresh: GazeLLE的in/out阈值
        yolo_conf: YOLO检测置信度阈值
    """
    
    # 初始化组件
    tracker = PersonTracker(
        iou_threshold=0.3, 
        max_disappear=30, 
        max_persons=max_persons
    )
    gaze_detector = GazeDetector(
        model=gazelle_model,
        transform=gazelle_transform,
        device=device,
        inout_threshold=inout_thresh,
        gaze_to_person_radius=0.15
    )
    graph = SocialMindGraph(fov_angle_deg=150)
    
    # 打开视频
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"无法打开视频: {video_path}")
        return
    
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f"视频信息: {width}x{height}, {fps} FPS, 总帧数: {total_frames}")
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    # 存储每帧的图结构（用于后续Module C分析）
    all_frame_graphs = []
    
    frame_count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_count += 1
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(rgb_frame)
        
        # ========== MODULE A1: Detection + Tracking ==========
        results = face_detector(rgb_frame, verbose=False, conf=yolo_conf)
        
        detected_boxes = []
        detected_confs = []
        if len(results) > 0 and len(results[0].boxes) > 0:
            boxes_tensor = results[0].boxes.xyxy.cpu().numpy()
            confs_tensor = results[0].boxes.conf.cpu().numpy()
            
            # 按面积排序，保留top-k
            areas = (boxes_tensor[:, 2] - boxes_tensor[:, 0]) * \
                    (boxes_tensor[:, 3] - boxes_tensor[:, 1])
            sorted_idx = np.argsort(areas)[::-1][:max_persons]
            
            for idx in sorted_idx:
                detected_boxes.append(boxes_tensor[idx].tolist())
                detected_confs.append(float(confs_tensor[idx]))
        
        # 更新Tracker → 获得带稳定ID的人物
        active_persons = tracker.update(
            detected_boxes, detected_confs,
            frame_count, width, height
        )
        
        # ========== MODULE A2: Gaze Detection ==========
        if len(active_persons) > 0:
            active_persons = gaze_detector.detect(pil_image, active_persons)
        
        # ========== MODULE B: Social Mind Graph ==========
        graph.build(active_persons, width, height)
        
        # 存储图结构
        frame_data = {
            "frame": frame_count,
            "graph": graph.to_dict()
        }
        all_frame_graphs.append(frame_data)
        
        # 可视化
        if len(active_persons) > 0:
            vis_image = visualize_with_ids(
                pil_image, active_persons, graph, inout_thresh
            )
            final_frame = cv2.cvtColor(np.array(vis_image), cv2.COLOR_RGB2BGR)
        else:
            final_frame = frame
        
        out.write(final_frame)
        
        # 日志
        if frame_count % 30 == 0:
            print(f"\n[Frame {frame_count}/{total_frames}]")
            print(f"  Active persons: {list(active_persons.keys())}")
            summary = graph.to_dict().get("summary", {})
            if summary:
                print(f"  Gaze interactions: {summary.get('gaze_interactions', [])}")
                print(f"  Mutual gaze: {summary.get('mutual_gaze_pairs', [])}")
            
            # 每隔一段时间打印完整图描述
            if frame_count % 150 == 0:
                print(graph.to_natural_language())
    
    cap.release()
    out.release()
    
    # 保存图结构JSON
    if json_output_path:
        # 采样保存（每秒1帧，避免文件过大）
        sampled = [fg for fg in all_frame_graphs if fg["frame"] % fps == 0]
        with open(json_output_path, 'w', encoding='utf-8') as f:
            json.dump(sampled, f, indent=2, default=str)
        print(f"Graph数据保存至: {json_output_path} ({len(sampled)} frames)")
    
    print(f"\n完成! 视频保存至: {output_path}")
    print(f"总处理帧数: {frame_count}")
    print(f"分配的Person ID数: {tracker.next_id - 1}")


# ================================================================
#                          运行
# ================================================================

video_input_path = "/Users/zehao/Desktop/HAI/MultiParty/Social-IQ/raw/vision/raw/_0at8kXKWSw_trimmed-out_copy.mp4"
video_output_path = "/Users/zehao/Desktop/HAI/MultiParty/code/output_pipeline.mp4"
json_output_path = "/Users/zehao/Desktop/HAI/MultiParty/code/output_graph.json"

os.makedirs(os.path.dirname(video_output_path), exist_ok=True)

process_video_full_pipeline(
    video_input_path,
    video_output_path,
    json_output_path=json_output_path,
    max_persons=5,
    inout_thresh=0.5,
    yolo_conf=0.7
)


In [20]:
# ================================================================
#  MODULE C: Symbolic ToM Reasoning Engine
#  MODULE D: LLM-Grounded Reasoning & Verbalization
#
#  接续 Module A1, A2, B 的代码
# ================================================================

import os
import json
import math
import re
import time
import base64
from io import BytesIO
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple, Any
from collections import defaultdict
from PIL import Image
import cv2
import numpy as np

# ================= API 配置 (请自行填写) =================

API_KEY = "sk-bbf9f24ff4194920a43e15749a2dad29"           # ← 填写你的API Key
API_BASE = "https://dashscope.aliyuncs.com/compatible-mode/v1"  # ← 填写你的API Base URL
MODEL_NAME = "qwen3.6-plus"
# 视觉模型是Qwen-VL-3.6，语言模型是qwen-plus-latest


import openai
client = openai.OpenAI(api_key=API_KEY, base_url=API_BASE)


# ================================================================
#               MODULE C: Symbolic ToM Reasoning Engine
# ================================================================

@dataclass
class MentalStateEntry:
    """心智状态知识库中的一条条目"""
    entry_id: str                    # 唯一标识
    order: int                       # ToM阶数 (0, 1, 2, 3...)
    entry_type: str                  # OBSERVABLE / BELIEF / IGNORANCE /
                                     # FEELING / INTENT / RECURSIVE_BELIEF /
                                     # SOCIAL_DYNAMIC
    subject: int                     # 持有该心智状态的person_id
    predicate: str                   # 心智状态描述
    about: Optional[int] = None      # 关于哪个person (如果有)
    value: bool = True               # True/False
    confidence: float = 1.0
    rule_applied: str = ""           # 使用了哪条规则
    premises: List[str] = field(default_factory=list)  # 前提条件的entry_ids
    natural_language: str = ""       # 自然语言表述


class MentalStateKnowledgeBase:
    """
    心智状态知识库 (MSKB)
    
    存储Module C所有推理结果，按阶数组织
    """
    
    def __init__(self):
        self.entries: Dict[str, MentalStateEntry] = {}
        self.by_order: Dict[int, List[str]] = defaultdict(list)
        self.by_subject: Dict[int, List[str]] = defaultdict(list)
        self._counter = 0
    
    def add(self, entry: MentalStateEntry) -> str:
        """添加一条心智状态条目，返回entry_id"""
        self._counter += 1
        entry.entry_id = f"MS_{entry.order}_{self._counter}"
        self.entries[entry.entry_id] = entry
        self.by_order[entry.order].append(entry.entry_id)
        self.by_subject[entry.subject].append(entry.entry_id)
        return entry.entry_id
    
    def get_by_order(self, order: int) -> List[MentalStateEntry]:
        return [self.entries[eid] for eid in self.by_order.get(order, [])]
    
    def get_by_subject(self, subject: int) -> List[MentalStateEntry]:
        return [self.entries[eid] for eid in self.by_subject.get(subject, [])]
    
    def to_dict(self) -> dict:
        result = {}
        for order in sorted(self.by_order.keys()):
            result[f"order_{order}"] = []
            for eid in self.by_order[order]:
                e = self.entries[eid]
                result[f"order_{order}"].append({
                    "id": e.entry_id,
                    "type": e.entry_type,
                    "subject": f"P{e.subject}",
                    "about": f"P{e.about}" if e.about else None,
                    "predicate": e.predicate,
                    "value": e.value,
                    "confidence": round(e.confidence, 3),
                    "rule": e.rule_applied,
                    "premises": e.premises,
                    "natural_language": e.natural_language
                })
        return result
    
    def to_natural_language(self, min_confidence: float = 0.3) -> str:
        """将MSKB转化为结构化的自然语言描述"""
        lines = []
        lines.append("=" * 60)
        lines.append("  MENTAL STATE KNOWLEDGE BASE")
        lines.append("=" * 60)
        
        for order in sorted(self.by_order.keys()):
            order_names = {
                0: "Observable States (what can be seen)",
                1: "1st-Order Beliefs (what each person knows/feels/wants)",
                2: "2nd-Order Beliefs (what each person thinks others know)",
                3: "3rd-Order Beliefs (recursive meta-beliefs)"
            }
            header = order_names.get(order, f"Order-{order} Mental States")
            lines.append(f"\n--- {header} ---")
            
            entries = [
                self.entries[eid] for eid in self.by_order[order]
                if self.entries[eid].confidence >= min_confidence
            ]
            
            if not entries:
                lines.append("  (none)")
                continue
            
            for e in entries:
                conf_label = "certain" if e.confidence > 0.8 else \
                            "likely" if e.confidence > 0.6 else \
                            "possible" if e.confidence > 0.4 else "uncertain"
                
                marker = "✓" if e.value else "✗"
                lines.append(f"  {marker} [{conf_label}] {e.natural_language}")
                if e.rule_applied:
                    lines.append(f"      Rule: {e.rule_applied}")
        
        # 添加社交动态总结
        dynamics = [
            self.entries[eid] for eid in self.entries
            if self.entries[eid].entry_type == "SOCIAL_DYNAMIC"
        ]
        if dynamics:
            lines.append(f"\n--- Social Dynamics ---")
            for d in dynamics:
                lines.append(f"  ★ {d.natural_language}")
        
        return "\n".join(lines)


class SymbolicToMEngine:
    """
    符号ToM推理引擎
    
    基于Social Mind Graph，通过可解释的规则，
    递归推导每个参与者的心智状态。
    
    每条规则产出：
      Observable Signal → Symbolic Rule → Mental State
      全程可追溯、可验证、可解释
    """
    
    def __init__(self, max_order: int = 3, 
                 confidence_decay: float = 0.85):
        """
        Args:
            max_order: 最大ToM推理阶数
            confidence_decay: 每增加一阶，置信度乘以此因子
        """
        self.max_order = max_order
        self.confidence_decay = confidence_decay
        self.mskb = MentalStateKnowledgeBase()
    
    def reason(self, graph: 'SocialMindGraph') -> MentalStateKnowledgeBase:
        """
        主推理函数：从Social Mind Graph推导完整的心智状态知识库
        
        Args:
            graph: Module B构建的Social Mind Graph
        
        Returns:
            MentalStateKnowledgeBase: 包含所有阶次推理结果
        """
        self.mskb = MentalStateKnowledgeBase()
        self.graph = graph
        
        if len(graph.nodes) < 1:
            return self.mskb
        
        # ========== 0阶：观察层 ==========
        self._reason_order_0()
        
        if len(graph.nodes) < 2:
            return self.mskb
        
        # ========== 1阶：信念/意图/感受 ==========
        self._reason_order_1()
        
        # ========== 2阶：递归信念 ==========
        self._reason_order_2()
        
        # ========== 高阶：递归展开 ==========
        for order in range(3, self.max_order + 1):
            new_entries = self._reason_higher_order(order)
            if new_entries == 0:
                break  # 无新推论，达到不动点
        
        # ========== 社交动态推断 ==========
        self._infer_social_dynamics()
        
        return self.mskb
    
    # ============================================================
    #  0阶推理：直接编码可观察状态
    # ============================================================
    
    def _reason_order_0(self):
        """0阶：无需推理，直接从图中编码每个人的可观察状态"""
        
        for pid, node in self.graph.nodes.items():
            gaze_info = node.observable_state.get("gaze", {})
            target_person = gaze_info.get("target_person")
            target_type = gaze_info.get("target_type", "unknown")
            
            # 编码注视行为
            if target_person is not None:
                self.mskb.add(MentalStateEntry(
                    entry_id="", order=0,
                    entry_type="OBSERVABLE",
                    subject=pid,
                    predicate=f"gazes_at_P{target_person}",
                    about=target_person,
                    value=True,
                    confidence=node.confidence.get("gaze", 0.5),
                    rule_applied="Direct Observation",
                    natural_language=f"P{pid} is looking at P{target_person}"
                ))
            elif target_type == "object":
                self.mskb.add(MentalStateEntry(
                    entry_id="", order=0,
                    entry_type="OBSERVABLE",
                    subject=pid,
                    predicate="gazes_at_object",
                    value=True,
                    confidence=node.confidence.get("gaze", 0.5),
                    rule_applied="Direct Observation",
                    natural_language=f"P{pid} is looking at an object/location (not at any person)"
                ))
            elif target_type == "out_of_frame":
                self.mskb.add(MentalStateEntry(
                    entry_id="", order=0,
                    entry_type="OBSERVABLE",
                    subject=pid,
                    predicate="gazes_out_of_frame",
                    value=True,
                    confidence=node.confidence.get("gaze", 0.5),
                    rule_applied="Direct Observation",
                    natural_language=f"P{pid} is looking outside the frame"
                ))
            
            # 编码互动模式
            # 被谁注视
            watchers = self.graph.gazed_by.get(pid, [])
            if watchers:
                self.mskb.add(MentalStateEntry(
                    entry_id="", order=0,
                    entry_type="OBSERVABLE",
                    subject=pid,
                    predicate=f"gazed_by_{len(watchers)}_people",
                    value=True,
                    confidence=0.8,
                    rule_applied="Aggregation",
                    natural_language=f"P{pid} is being looked at by {len(watchers)} person(s): {['P'+str(w) for w in watchers]}"
                ))
    
    # ============================================================
    #  1阶推理：信念、意图、感受
    # ============================================================
    
    def _reason_order_1(self):
        """1阶ToM：从可观察状态推导每个人的信念/意图/感受"""
        
        pids = list(self.graph.nodes.keys())
        
        for pid_i in pids:
            for pid_j in pids:
                if pid_i == pid_j:
                    continue
                
                va = self.graph.visual_access_matrix.get((pid_i, pid_j))
                if va is None:
                    continue
                
                can_observe = va["can_observe"]
                va_conf = va["confidence"]
                
                # === Rule 1.1: Visual Access → Awareness ===
                # 如果P_i能看到P_j，且P_j在做某事，则P_i知道P_j在做该事
                
                j_gaze_target = self.graph.gazes_at.get(pid_j)
                j_node = self.graph.nodes.get(pid_j)
                
                if can_observe and j_node:
                    j_gaze_type = j_node.observable_state.get("gaze", {}).get("target_type", "unknown")
                    
                    if j_gaze_target is not None:
                        # P_j在看某人
                        conf = va_conf * j_node.confidence.get("gaze", 0.5)
                        self.mskb.add(MentalStateEntry(
                            entry_id="", order=1,
                            entry_type="BELIEF",
                            subject=pid_i,
                            predicate=f"aware_that_P{pid_j}_looks_at_P{j_gaze_target}",
                            about=pid_j,
                            value=True,
                            confidence=conf,
                            rule_applied="Rule 1.1: Visual Access → Awareness",
                            natural_language=f"P{pid_i} can see that P{pid_j} is looking at P{j_gaze_target}"
                        ))
                    elif j_gaze_type == "object":
                        conf = va_conf * 0.7
                        self.mskb.add(MentalStateEntry(
                            entry_id="", order=1,
                            entry_type="BELIEF",
                            subject=pid_i,
                            predicate=f"aware_that_P{pid_j}_looks_at_object",
                            about=pid_j,
                            value=True,
                            confidence=conf,
                            rule_applied="Rule 1.1: Visual Access → Awareness",
                            natural_language=f"P{pid_i} can see that P{pid_j} is looking at something (not a person)"
                        ))
                
                # === Rule 1.2: No Visual Access → Ignorance ===
                if not can_observe:
                    # P_i看不到P_j → P_i不知道P_j在做什么
                    if j_gaze_target is not None:
                        conf = va_conf
                        self.mskb.add(MentalStateEntry(
                            entry_id="", order=1,
                            entry_type="IGNORANCE",
                            subject=pid_i,
                            predicate=f"unaware_that_P{pid_j}_looks_at_P{j_gaze_target}",
                            about=pid_j,
                            value=True,
                            confidence=conf,
                            rule_applied="Rule 1.2: No Visual Access → Ignorance",
                            natural_language=f"P{pid_i} does NOT know that P{pid_j} is looking at P{j_gaze_target} (P{pid_j} is outside P{pid_i}'s field of view)"
                        ))
                    
                    # 特别重要：如果P_j在看P_i，但P_i看不到P_j
                    if j_gaze_target == pid_i:
                        conf = va_conf * self.graph.nodes[pid_j].confidence.get("gaze", 0.5)
                        self.mskb.add(MentalStateEntry(
                            entry_id="", order=1,
                            entry_type="IGNORANCE",
                            subject=pid_i,
                            predicate=f"unaware_of_being_watched_by_P{pid_j}",
                            about=pid_j,
                            value=True,
                            confidence=conf,
                            rule_applied="Rule 1.2b: No Access + Being Watched → Unaware of Surveillance",
                            natural_language=f"P{pid_i} does NOT know that P{pid_j} is watching them (P{pid_i} cannot see P{pid_j})"
                        ))
        
        # === Rule 1.3: Unreciprocated Engagement → Feeling ===
        for rel in self.graph.derived_relations:
            if rel.relation_type == "UNRECIPROCATED_ENGAGEMENT" and rel.value:
                pid_i = rel.participants[0]  # 在看对方的人
                pid_j = rel.participants[1]  # 没有回看的人
                
                # 检查P_i是否能看到P_j（P_i应该能看到，因为P_i在看P_j）
                va = self.graph.visual_access_matrix.get((pid_i, pid_j))
                if va and va["can_observe"]:
                    conf = rel.confidence * 0.7  # 感受推理置信度稍低
                    self.mskb.add(MentalStateEntry(
                        entry_id="", order=1,
                        entry_type="FEELING",
                        subject=pid_i,
                        predicate=f"may_feel_ignored_or_waiting_for_P{pid_j}",
                        about=pid_j,
                        value=True,
                        confidence=conf,
                        rule_applied="Rule 1.3: Unreciprocated Engagement → Possible Feeling",
                        natural_language=f"P{pid_i} is engaging P{pid_j} but P{pid_j} is not looking back — P{pid_i} may feel ignored or be waiting for P{pid_j}'s attention"
                    ))
        
        # === Rule 1.4: Mutual Gaze → Shared Engagement ===
        for rel in self.graph.derived_relations:
            if rel.relation_type == "MUTUAL_GAZE" and rel.value:
                pid_i, pid_j = rel.participants
                conf = rel.confidence
                
                self.mskb.add(MentalStateEntry(
                    entry_id="", order=1,
                    entry_type="BELIEF",
                    subject=pid_i,
                    predicate=f"mutually_engaged_with_P{pid_j}",
                    about=pid_j,
                    value=True,
                    confidence=conf,
                    rule_applied="Rule 1.4: Mutual Gaze → Shared Engagement",
                    natural_language=f"P{pid_i} and P{pid_j} are mutually engaged (both looking at each other)"
                ))
                self.mskb.add(MentalStateEntry(
                    entry_id="", order=1,
                    entry_type="BELIEF",
                    subject=pid_j,
                    predicate=f"mutually_engaged_with_P{pid_i}",
                    about=pid_i,
                    value=True,
                    confidence=conf,
                    rule_applied="Rule 1.4: Mutual Gaze → Shared Engagement",
                    natural_language=f"P{pid_j} and P{pid_i} are mutually engaged (both looking at each other)"
                ))
        
        # === Rule 1.5: Being Gazed At + Can Observe Gazer → Awareness ===
        for pid_i in pids:
            watchers = self.graph.gazed_by.get(pid_i, [])
            for watcher_pid in watchers:
                va = self.graph.visual_access_matrix.get((pid_i, watcher_pid))
                if va and va["can_observe"]:
                    conf = va["confidence"] * 0.8
                    self.mskb.add(MentalStateEntry(
                        entry_id="", order=1,
                        entry_type="BELIEF",
                        subject=pid_i,
                        predicate=f"aware_of_being_watched_by_P{watcher_pid}",
                        about=watcher_pid,
                        value=True,
                        confidence=conf,
                        rule_applied="Rule 1.5: Being Gazed At + Visual Access → Aware of Being Watched",
                        natural_language=f"P{pid_i} can see that P{watcher_pid} is looking at them, so P{pid_i} knows P{watcher_pid} is watching"
                    ))
        
        # === Rule 1.6: Excluded Position → Feeling ===
        for rel in self.graph.derived_relations:
            if rel.relation_type == "POSSIBLY_EXCLUDED" and rel.value:
                pid_i = rel.participants[0]
                conf = rel.confidence * 0.6
                self.mskb.add(MentalStateEntry(
                    entry_id="", order=1,
                    entry_type="FEELING",
                    subject=pid_i,
                    predicate="may_feel_excluded_from_group",
                    value=True,
                    confidence=conf,
                    rule_applied="Rule 1.6: No One Gazing At + No Reciprocated Engagement → Possibly Excluded",
                    natural_language=f"P{pid_i} is not being looked at by anyone and has no reciprocated interactions — may feel excluded"
                ))
    
    # ============================================================
    #  2阶推理：递归信念（A认为B知道/不知道什么）
    # ============================================================
    
    def _reason_order_2(self):
        """2阶ToM：基于1阶结果 + Visual Access，推导递归信念"""
        
        pids = list(self.graph.nodes.keys())
        order1_entries = self.mskb.get_by_order(1)
        
        for entry in order1_entries:
            if entry.entry_type not in ["BELIEF", "IGNORANCE", "FEELING"]:
                continue
            
            subject = entry.subject  # 持有该1阶信念的人
            
            # 对于每个其他人，判断他们是否能推知subject的这个信念
            for pid_k in pids:
                if pid_k == subject:
                    continue
                
                # === Rule 2.1: 知道对方知道什么 ===
                # 如果P_k能看到subject的信念的"证据"
                
                if entry.entry_type == "IGNORANCE" and entry.about is not None:
                    # entry: subject不知道某事 (因为看不到)
                    # 例: P1不知道P3在看自己
                    
                    # P_k能否推知"P_subject不知道这件事"？
                    # 条件: P_k能看到subject，且P_k能看到entry.about
                    # → P_k能同时看到两边 → P_k能推知信息不对称
                    
                    va_k_subject = self.graph.visual_access_matrix.get((pid_k, subject))
                    va_k_about = self.graph.visual_access_matrix.get((pid_k, entry.about))
                    va_subject_about = self.graph.visual_access_matrix.get((subject, entry.about))
                    
                    if (va_k_subject and va_k_subject["can_observe"] and
                        va_k_about and va_k_about["can_observe"] and
                        va_subject_about and not va_subject_about["can_observe"]):
                        
                        conf = entry.confidence * self.confidence_decay
                        conf *= min(va_k_subject["confidence"], va_k_about["confidence"])
                        
                        self.mskb.add(MentalStateEntry(
                            entry_id="", order=2,
                            entry_type="RECURSIVE_BELIEF",
                            subject=pid_k,
                            predicate=f"knows_that_P{subject}_{entry.predicate}",
                            about=subject,
                            value=True,
                            confidence=conf,
                            rule_applied="Rule 2.1: Observer can see both parties → knows about information asymmetry",
                            premises=[entry.entry_id],
                            natural_language=f"P{pid_k} can observe both P{subject} and P{entry.about}, and can tell that P{subject} is unaware of P{entry.about}'s behavior"
                        ))
                
                # === Rule 2.2: 被观察者知道/不知道自己被观察 ===
                if entry.entry_type == "BELIEF" and "aware_of_being_watched" in entry.predicate:
                    # entry: subject知道自己被某人看
                    # P_k是否知道subject知道？
                    watcher = entry.about
                    
                    va_k_subject = self.graph.visual_access_matrix.get((pid_k, subject))
                    
                    if va_k_subject and va_k_subject["can_observe"]:
                        # P_k能看到subject → 可能能推知subject的反应
                        conf = entry.confidence * self.confidence_decay * va_k_subject["confidence"]
                        self.mskb.add(MentalStateEntry(
                            entry_id="", order=2,
                            entry_type="RECURSIVE_BELIEF",
                            subject=pid_k,
                            predicate=f"may_know_that_P{subject}_is_aware_of_P{watcher}_watching",
                            about=subject,
                            value=True,
                            confidence=conf * 0.7,  # 更不确定
                            rule_applied="Rule 2.2: Observer sees subject's awareness of being watched",
                            premises=[entry.entry_id],
                            natural_language=f"P{pid_k} can observe P{subject}, and may realize that P{subject} knows P{watcher} is watching"
                        ))
                
                # === Rule 2.3: 知道对方的感受 ===
                if entry.entry_type == "FEELING":
                    va_k_subject = self.graph.visual_access_matrix.get((pid_k, subject))
                    
                    if va_k_subject and va_k_subject["can_observe"]:
                        conf = entry.confidence * self.confidence_decay * va_k_subject["confidence"] * 0.6
                        self.mskb.add(MentalStateEntry(
                            entry_id="", order=2,
                            entry_type="RECURSIVE_BELIEF",
                            subject=pid_k,
                            predicate=f"may_sense_that_P{subject}_feels_{entry.predicate.split('feel_')[-1] if 'feel_' in entry.predicate else 'something'}",
                            about=subject,
                            value=True,
                            confidence=conf,
                            rule_applied="Rule 2.3: Observer can see subject → may sense their feeling",
                            premises=[entry.entry_id],
                            natural_language=f"P{pid_k} can observe P{subject} and may sense that P{subject} {entry.predicate.replace('_', ' ')}"
                        ))
        
        # === Rule 2.4: Secret Observer Detection ===
        # 如果P_i在看P_j，但P_j看不到P_i → P_i知道P_j不知道被看
        for pid_i in pids:
            target = self.graph.gazes_at.get(pid_i)
            if target is None:
                continue
            
            va_target_i = self.graph.visual_access_matrix.get((target, pid_i))
            if va_target_i and not va_target_i["can_observe"]:
                # target看不到pid_i → pid_i是"秘密观察者"
                conf = va_target_i["confidence"] * 0.8
                self.mskb.add(MentalStateEntry(
                    entry_id="", order=2,
                    entry_type="RECURSIVE_BELIEF",
                    subject=pid_i,
                    predicate=f"knows_P{target}_is_unaware_of_being_watched",
                    about=target,
                    value=True,
                    confidence=conf,
                    rule_applied="Rule 2.4: Secret Observer (I watch you, you can't see me)",
                    natural_language=f"P{pid_i} knows that P{target} is unaware of being watched by P{pid_i}, because P{target} cannot see P{pid_i}"
                ))
    
    # ============================================================
    #  高阶推理：递归展开
    # ============================================================
    
    def _reason_higher_order(self, order: int) -> int:
        """
        n阶推理：在(n-1)阶结果上再应用Visual Access规则
        
        Returns: 新产生的条目数量
        """
        pids = list(self.graph.nodes.keys())
        prev_entries = self.mskb.get_by_order(order - 1)
        new_count = 0
        
        for entry in prev_entries:
            if entry.entry_type != "RECURSIVE_BELIEF":
                continue
            if entry.confidence < 0.3:
                continue  # 置信度太低，不再递归
            
            subject = entry.subject
            
            for pid_k in pids:
                if pid_k == subject:
                    continue
                
                va_k_subject = self.graph.visual_access_matrix.get((pid_k, subject))
                if va_k_subject is None:
                    continue
                
                conf = entry.confidence * self.confidence_decay * va_k_subject["confidence"]
                
                if conf < 0.2:
                    continue  # 置信度衰减到太低
                
                if va_k_subject["can_observe"]:
                    self.mskb.add(MentalStateEntry(
                        entry_id="", order=order,
                        entry_type="RECURSIVE_BELIEF",
                        subject=pid_k,
                        predicate=f"may_know_that_P{subject}_{entry.predicate}",
                        about=subject,
                        value=True,
                        confidence=conf,
                        rule_applied=f"Rule {order}.1: Higher-order recursive belief (order={order})",
                        premises=[entry.entry_id],
                        natural_language=f"P{pid_k} can observe P{subject}, and may realize that P{subject} {entry.predicate.replace('_', ' ')}"
                    ))
                    new_count += 1
        
        return new_count
    
    # ============================================================
    #  社交动态推断
    # ============================================================
    
    def _infer_social_dynamics(self):
        """基于所有阶次的推理结果，推断高层社交动态"""
        
        pids = list(self.graph.nodes.keys())
        
        # Dynamic 1: Information Advantage
        # 找到拥有信息优势的人
        for entry in self.mskb.get_by_order(2):
            if "unaware" in entry.predicate and entry.value:
                self.mskb.add(MentalStateEntry(
                    entry_id="", order=0,
                    entry_type="SOCIAL_DYNAMIC",
                    subject=entry.subject,
                    predicate="has_information_advantage",
                    about=entry.about,
                    value=True,
                    confidence=entry.confidence,
                    rule_applied="Dynamic: Information Asymmetry Detection",
                    premises=[entry.entry_id],
                    natural_language=f"P{entry.subject} has an information advantage over P{entry.about} — P{entry.subject} knows something that P{entry.about} doesn't know"
                ))
        
        # Dynamic 2: Attention Center
        for rel in self.graph.derived_relations:
            if rel.relation_type == "ATTENTION_CENTER":
                center_pid = rel.participants[0]
                self.mskb.add(MentalStateEntry(
                    entry_id="", order=0,
                    entry_type="SOCIAL_DYNAMIC",
                    subject=center_pid,
                    predicate="is_attention_center",
                    value=True,
                    confidence=rel.confidence,
                    rule_applied="Dynamic: Attention Center Detection",
                    natural_language=f"P{center_pid} is the center of attention — being looked at by multiple people"
                ))
        
        # Dynamic 3: Social Isolation
        for rel in self.graph.derived_relations:
            if rel.relation_type == "POSSIBLY_EXCLUDED":
                iso_pid = rel.participants[0]
                self.mskb.add(MentalStateEntry(
                    entry_id="", order=0,
                    entry_type="SOCIAL_DYNAMIC",
                    subject=iso_pid,
                    predicate="is_possibly_isolated",
                    value=True,
                    confidence=rel.confidence,
                    rule_applied="Dynamic: Social Isolation Detection",
                    natural_language=f"P{iso_pid} appears socially isolated — not being engaged by others"
                ))


# ================================================================
#               MODULE D: LLM-Grounded Reasoning
# ================================================================

class LLMReasoningModule:
    """
    Module D: LLM辅助推理与问答
    
    职责：
    D1: 补充规则无法覆盖的模糊推理
    D2: 回答用户的具体Query
    D3: 生成综合性的社交场景分析
    
    关键原则：
    - LLM是翻译者和补充者，不是推理者
    - LLM的推理必须以符号推理结果为约束
    - 明确区分 verified facts vs inferred opinions
    """
    
    def __init__(self, model_name: str = MODEL_NAME,
                 max_retries: int = 3):
        self.model_name = model_name
        self.max_retries = max_retries
    
    def _encode_image(self, pil_image: Image.Image, 
                       max_size: int = 512) -> str:
        """将PIL图像编码为base64字符串"""
        # 缩放以减少token消耗
        w, h = pil_image.size
        if max(w, h) > max_size:
            ratio = max_size / max(w, h)
            pil_image = pil_image.resize(
                (int(w * ratio), int(h * ratio)), 
                Image.LANCZOS
            )
        
        buffer = BytesIO()
        pil_image.save(buffer, format="JPEG", quality=85)
        return base64.b64encode(buffer.getvalue()).decode("utf-8")
    
    def _call_llm(self, messages: list, temperature: float = 0.3,
                   max_tokens: int = 2000) -> str:
        """调用LLM API，带重试逻辑"""
        for attempt in range(self.max_retries):
            try:
                response = client.chat.completions.create(
                    model=self.model_name,
                    messages=messages,
                    temperature=temperature,
                    max_tokens=max_tokens
                )
                return response.choices[0].message.content
            except Exception as e:
                print(f"  LLM API call failed (attempt {attempt+1}): {e}")
                if attempt < self.max_retries - 1:
                    time.sleep(2 ** attempt)
                else:
                    return f"[LLM Error: {str(e)}]"
    
    def _call_llm_with_image(self, system_prompt: str, 
                              user_prompt: str,
                              pil_image: Image.Image,
                              temperature: float = 0.3,
                              max_tokens: int = 2000) -> str:
        """调用多模态LLM API（带图像）"""
        img_b64 = self._encode_image(pil_image)
        
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": [
                {"type": "image_url", "image_url": {
                    "url": f"data:image/jpeg;base64,{img_b64}"
                }},
                {"type": "text", "text": user_prompt}
            ]}
        ]
        
        return self._call_llm(messages, temperature, max_tokens)
    
    # ============================================================
    #  D1: 模糊推理补充
    # ============================================================
    
    def complementary_reasoning(self, 
                                 mskb: MentalStateKnowledgeBase,
                                 graph: 'SocialMindGraph',
                                 pil_image: Image.Image) -> str:
        """
        补充规则无法覆盖的模糊推理
        
        LLM在verified facts约束下，推理：
        - 具体的意图（不只是"感到被忽视"，而是"可能想讨论某事"）
        - 场景上下文（教室？会议？社交聚会？）
        - 文化/社交规范相关的推断
        """
        system_prompt = """You are a social cognition expert analyzing multi-party social scenes.

You will be given:
1. An image of a social scene
2. Formally verified mental states (derived by symbolic reasoning from visual social signals like gaze direction)
3. The Social Mind Graph description

Your task: Provide complementary reasoning that the symbolic rules cannot cover.
Focus on:
- Scene context (what kind of social setting is this?)
- Specific intentions (why might someone be behaving this way?)
- Social dynamics that require world knowledge

CRITICAL RULES:
- Do NOT contradict any verified facts
- Clearly label your inferences as [INFERRED] vs verified facts as [VERIFIED]
- Express uncertainty when appropriate
- Be concise and structured"""

        user_prompt = f"""
=== SOCIAL MIND GRAPH ===
{graph.to_natural_language()}

=== VERIFIED MENTAL STATES (from symbolic reasoning) ===
{mskb.to_natural_language()}

=== YOUR TASK ===
Based on the image and the verified facts above, provide:

1. SCENE CONTEXT: What kind of social setting is this? (e.g., classroom, meeting, casual conversation)

2. CONTEXTUAL INTENT: For each person, what might be their specific social goal given the context?

3. UNRESOLVED QUESTIONS: What aspects of the social dynamics cannot be determined from gaze alone and would require additional information (audio, facial expression details, etc.)?

4. OVERALL SOCIAL DYNAMIC SUMMARY: In 2-3 sentences, describe the social dynamic of this scene, integrating both verified facts and your contextual inferences.
"""
        
        return self._call_llm_with_image(system_prompt, user_prompt, pil_image)
    
    # ============================================================
    #  D2: Query Answering (核心功能)
    # ============================================================
    
    def answer_query(self,
                     query: str,
                     mskb: MentalStateKnowledgeBase,
                     graph: 'SocialMindGraph',
                     pil_image: Image.Image,
                     complementary_analysis: str = "") -> dict:
        """
        回答用户的具体问题
        
        关键设计：LLM回答时必须引用符号推理的verified facts作为证据
        
        Args:
            query: 用户问题
            mskb: 心智状态知识库
            graph: 社交心智图
            pil_image: 场景图片
            complementary_analysis: D1的补充分析结果
        
        Returns:
            {
                "answer": str,
                "evidence_chain": list,
                "confidence": str,
                "tom_order_used": int
            }
        """
        system_prompt = """You are an expert social scene analyst using a neuro-symbolic reasoning framework.

You will be given:
1. An image of a multi-party social scene
2. A Social Mind Graph (who can see whom, who is looking at whom)
3. Formally VERIFIED mental states derived by symbolic Theory of Mind rules
4. Contextual analysis from a complementary reasoning module
5. A specific question to answer

YOUR RESPONSE FORMAT (strictly follow this JSON structure):
{
    "answer": "Your answer to the question",
    "evidence_chain": [
        {"type": "verified", "fact": "description of verified fact", "rule": "which rule produced this"},
        {"type": "inferred", "fact": "description of inferred fact", "basis": "what this is based on"}
    ],
    "confidence": "high/medium/low",
    "tom_order_used": 0,
    "reasoning": "Step-by-step reasoning trace"
}

CRITICAL RULES:
- Always ground your answer in VERIFIED facts when possible
- Clearly distinguish [VERIFIED by symbolic reasoning] vs [INFERRED by contextual reasoning]
- Report the highest ToM order used in your reasoning (0=observation, 1=belief/intent, 2=recursive belief, 3+=higher)
- If the question cannot be answered from available evidence, say so honestly
- Consider what each person CAN and CANNOT see (Visual Access Matrix) — this is crucial for ToM questions"""

        verified_facts = mskb.to_natural_language(min_confidence=0.3)
        graph_desc = graph.to_natural_language()
        
        user_prompt = f"""
=== SOCIAL MIND GRAPH ===
{graph_desc}

=== VERIFIED MENTAL STATES ===
{verified_facts}

=== CONTEXTUAL ANALYSIS ===
{complementary_analysis if complementary_analysis else "(not yet performed)"}

=== QUESTION ===
{query}

Please answer the question following the required JSON format. Ground your answer in the verified facts and visual evidence.
"""
        
        raw_response = self._call_llm_with_image(
            system_prompt, user_prompt, pil_image,
            temperature=0.2, max_tokens=1500
        )
        
        # 尝试解析JSON
        result = self._parse_response(raw_response, query)
        return result
    
    def _parse_response(self, raw_response: str, query: str) -> dict:
        """解析LLM返回的JSON（带容错）"""
        # 尝试从response中提取JSON
        try:
            # 尝试直接解析
            json_match = re.search(r'\{[\s\S]*\}', raw_response)
            if json_match:
                result = json.loads(json_match.group())
                # 确保所有必要字段存在
                result.setdefault("answer", "")
                result.setdefault("evidence_chain", [])
                result.setdefault("confidence", "medium")
                result.setdefault("tom_order_used", 0)
                result.setdefault("reasoning", "")
                return result
        except json.JSONDecodeError:
            pass
        
        # JSON解析失败，返回原始文本
        return {
            "answer": raw_response,
            "evidence_chain": [],
            "confidence": "unknown",
            "tom_order_used": -1,
            "reasoning": "Failed to parse structured response"
        }
    
    # ============================================================
    #  D3: Social-IQ Format QA
    # ============================================================
    
    def answer_social_iq(self,
                         question: str,
                         answer_choices: List[dict],
                         mskb: MentalStateKnowledgeBase,
                         graph: 'SocialMindGraph',
                         pil_image: Image.Image,
                         complementary_analysis: str = "") -> dict:
        """
        专门针对Social-IQ格式的多选题回答
        
        Args:
            question: 问题文本
            answer_choices: [{"label": "a"/"i", "text": "..."}]
            mskb: 心智状态知识库
            graph: 社交心智图
            pil_image: 场景图片
            complementary_analysis: D1的补充分析
        
        Returns:
            {"selected_answer": str, "reasoning": str, ...}
        """
        system_prompt = """You are an expert social scene analyst using a neuro-symbolic Theory of Mind framework.

You will be given a multi-party social scene with:
1. Visual evidence (image)
2. Formally verified mental states from symbolic ToM reasoning
3. A multiple-choice question about social dynamics

Your task: Select the best answer and explain your reasoning.

RESPONSE FORMAT (JSON):
{
    "selected_answer_index": 0,
    "selected_answer_text": "the answer text",
    "reasoning": "step-by-step reasoning grounded in verified facts",
    "evidence_used": ["list of key verified facts used"],
    "tom_order_required": 0,
    "confidence": "high/medium/low"
}

RULES:
- Ground reasoning in VERIFIED mental states when relevant
- Consider Visual Access (who can see whom) for perspective-taking questions
- Analyze each answer choice independently based on the evidence; do not assume a specific option index is inherently correct or incorrect."""

        # 格式化选项
        choices_text = "\n".join([
            f"  [{'index'}] {c['text']}"
            # for i, c in enumerate(answer_choices)
            for c in answer_choices
        ])
        
        verified_facts = mskb.to_natural_language(min_confidence=0.3)
        graph_desc = graph.to_natural_language()
        
        user_prompt = f"""
=== SOCIAL MIND GRAPH ===
{graph_desc}

=== VERIFIED MENTAL STATES ===
{verified_facts}

=== CONTEXTUAL ANALYSIS ===
{complementary_analysis if complementary_analysis else "(not available)"}

=== QUESTION ===
{question}

=== ANSWER CHOICES ===
{choices_text}

Select the best answer (by index) and explain your reasoning.
"""
        
        raw_response = self._call_llm_with_image(
            system_prompt, user_prompt, pil_image,
            temperature=0.1, max_tokens=1500
        )
        
        result = self._parse_response(raw_response, question)
        result["question"] = question
        result["choices"] = answer_choices
        return result


# ================================================================
#          Social-IQ 数据解析器
# ================================================================

class SocialIQParser:
    """解析Social-IQ数据集的QA文件"""
    
    @staticmethod
    # def parse_qa_file(filepath: str) -> List[dict]:
    #     """
    #     解析Social-IQ的QA文本文件
        
    #     格式：
    #     q1: question text
    #     a: correct answer
    #     i: incorrect answer
    #     a: correct answer (detailed)
    #     ...
        
    #     Returns:
    #         [{
    #             "question_id": "q1",
    #             "question": "...",
    #             "answers": [{"label": "a"/"i", "text": "..."}]
    #         }, ...]
    #     """
    #     questions = []
    #     current_q = None
        
    #     with open(filepath, 'r', encoding='utf-8') as f:
    #         for line in f:
    #             line = line.strip()
    #             if not line:
    #                 continue
                
    #             if line.startswith('q') and ':' in line:
    #                 # 新问题
    #                 if current_q is not None:
    #                     questions.append(current_q)
                    
    #                 parts = line.split(':', 1)
    #                 q_id = parts[0].strip()
    #                 q_text = parts[1].strip() if len(parts) > 1 else ""
                    
    #                 current_q = {
    #                     "question_id": q_id,
    #                     "question": q_text,
    #                     "answers": []
    #                 }
                
    #             elif line.startswith('a:') or line.startswith('i:'):
    #                 if current_q is not None:
    #                     label = line[0]  # 'a' or 'i'
    #                     text = line[2:].strip()
    #                     current_q["answers"].append({
    #                         "label": label,
    #                         "text": text
    #                     })
        
    #     if current_q is not None:
    #         questions.append(current_q)
        
    #     return questions


    def parse_qa_file(filepath: str, target_vid_name: str = None) -> List[dict]:
        """
        解析Social-IQ 2.0 的 JSON/JSONL QA文件
        
        Args:
            filepath: json/jsonl 文件的路径
            target_vid_name: 可选。如果传入的是一个包含所有视频的大json文件，
                             可以通过该参数只筛选出当前测试视频的 QA。
        """
        questions =[]
        
        with open(filepath, 'r', encoding='utf-8') as f:
            content = f.read().strip()
            
            # 兼容处理：检查是标准JSON列表 还是 JSON Lines(每一行是一个独立JSON)
            if content.startswith('['):
                data_list = json.loads(content)
            else:
                data_list =[json.loads(line) for line in content.split('\n') if line.strip()]
        
        for data in data_list:
            # 如果指定了 vid_name，过滤掉不属于该视频的QA
            if target_vid_name and data.get("vid_name") != target_vid_name:
                continue
            
            q_id = data.get("qid")
            q_text = data.get("q")
            correct_idx = data.get("answer_idx", -1)
            
            current_q = {
                "question_id": q_id,
                "question": q_text,
                "vid_name": data.get("vid_name"),
                "answers": [],
                "correct_indices": [],
                "incorrect_indices":[]
            }
            
            # Social-IQ 2 默认有4个选项 a0, a1, a2, a3
            for i in range(4):
                ans_key = f"a{i}"
                if ans_key in data:
                    text = data[ans_key]
                    is_correct = (i == correct_idx)
                    label = 'a' if is_correct else 'i'  # 兼容旧代码的 label
                    
                    current_q["answers"].append({
                        "index": i, 
                        "label": label,
                        "text": text, 
                        "is_correct": is_correct
                    })
                    
                    if is_correct:
                        current_q["correct_indices"].append(i)
                    else:
                        current_q["incorrect_indices"].append(i)
            
            questions.append(current_q)
            
        return questions


# ================================================================
#          完整Pipeline整合
# ================================================================

class SocialMindPipeline:
    """
    完整的Neuro-Symbolic Social Mind Reasoning Pipeline
    
    Module A (Neural Perception) → Module B (Graph) →
    Module C (Symbolic ToM) → Module D (LLM Reasoning)
    """
    
    def __init__(self, 
                 gazelle_model, gazelle_transform, device,
                 face_detector,
                 max_persons=5, inout_thresh=0.5, yolo_conf=0.7,
                 max_tom_order=3):
        
        # Module A
        self.tracker = PersonTracker(
            iou_threshold=0.3, max_disappear=30, max_persons=max_persons
        )
        self.gaze_detector = GazeDetector(
            model=gazelle_model, transform=gazelle_transform,
            device=device, inout_threshold=inout_thresh
        )
        self.yolo = face_detector
        self.yolo_conf = yolo_conf
        self.max_persons = max_persons
        
        # Module B
        self.graph = SocialMindGraph(fov_angle_deg=150)
        
        # Module C
        self.tom_engine = SymbolicToMEngine(max_order=max_tom_order)
        
        # Module D
        self.llm_module = LLMReasoningModule()
        
        # 存储
        self.frame_results = []
    
    def process_video_and_answer(self,
                                  video_path: str,
                                  qa_file_path: str,
                                  output_video_path: str = None,
                                  output_json_path: str = None,
                                  sample_frames: List[int] = None,
                                  keyframe_interval: int = 30) -> dict:
        """
        完整Pipeline：处理视频 + 回答Social-IQ问题
        
        Args:
            video_path: 输入视频
            qa_file_path: Social-IQ QA文件路径
            output_video_path: 输出可视化视频路径
            output_json_path: 输出完整结果的JSON路径
            sample_frames: 指定分析的帧号列表（None则自动采样）
            keyframe_interval: 自动采样间隔（仅sample_frames=None时生效）
        
        Returns:
            完整结果字典
        """
        
        # 1. 解析QA文件
        print("=" * 60)
        print("STEP 1: Parsing Social-IQ QA file")
        print("=" * 60)
        
        vid_name = os.path.splitext(os.path.basename(video_path))[0]
        qa_data = SocialIQParser.parse_qa_file(qa_file_path, target_vid_name=vid_name)
        # qa_data = SocialIQParser.parse_qa_file(qa_file_path)
        print(f"  Loaded {len(qa_data)} questions")
        for q in qa_data:
            print(f"    {q['question_id']}: {q['question'][:60]}...")
        
        # 2. 处理视频，收集关键帧的Social Mind Graph
        print("\n" + "=" * 60)
        print("STEP 2: Processing video (Module A + B + C)")
        print("=" * 60)
        
        keyframe_data = self._process_video_collect_keyframes(
            video_path, output_video_path,
            sample_frames, keyframe_interval
        )
        
        print(f"\n  Collected {len(keyframe_data)} keyframes for analysis")
        
        # 3. 选择最具代表性的关键帧（人最多、互动最丰富的帧）
        print("\n" + "=" * 60)
        print("STEP 3: Selecting representative keyframe")
        print("=" * 60)
        
        best_frame = self._select_best_keyframe(keyframe_data)
        
        if best_frame is None:
            print("  WARNING: No valid keyframe found!")
            return {"error": "No valid keyframes with detected persons"}
        
        print(f"  Selected frame {best_frame['frame_idx']} "
              f"({best_frame['num_persons']} persons, "
              f"{best_frame['num_interactions']} interactions)")
        
        # 4. 在最佳关键帧上运行Module D
        print("\n" + "=" * 60)
        print("STEP 4: LLM-Grounded Reasoning (Module D)")
        print("=" * 60)
        
        pil_image = best_frame["pil_image"]
        mskb = best_frame["mskb"]
        graph = best_frame["graph"]
        
        # D1: 补充推理
        print("\n  [D1] Running complementary reasoning...")
        complementary = self.llm_module.complementary_reasoning(
            mskb, graph, pil_image
        )
        print(f"  Complementary analysis:\n{complementary[:500]}...")
        
        # D2: 回答每个问题
        print("\n  [D2] Answering Social-IQ questions...")
        
        all_qa_results = []
        
        for q_data in qa_data:
            print(f"\n  --- {q_data['question_id']}: {q_data['question'][:60]}... ---")
            
            result = self.llm_module.answer_social_iq(
                question=q_data["question"],
                answer_choices=q_data["answers"],
                mskb=mskb,
                graph=graph,
                pil_image=pil_image,
                complementary_analysis=complementary
            )
            
            # 打印结果
            if "selected_answer_index" in result:
                idx = result.get("selected_answer_index", -1)
                print(f"  Selected: [{idx}] {result.get('selected_answer_text', 'N/A')}")
            print(f"  Confidence: {result.get('confidence', 'N/A')}")
            print(f"  ToM order: {result.get('tom_order_required', 'N/A')}")
            reasoning = result.get('reasoning', '')
            if reasoning:
                print(f"  Reasoning: {reasoning[:200]}...")
            
            all_qa_results.append({
                "question_id": q_data["question_id"],
                "question": q_data["question"],
                "ground_truth_answers": q_data["answers"],
                "model_response": result
            })
        
        # 5. 汇总结果
        print("\n" + "=" * 60)
        print("STEP 5: Compiling results")
        print("=" * 60)
        
        full_result = {
            "video_path": video_path,
            "qa_file": qa_file_path,
            "keyframe_info": {
                "frame_idx": best_frame["frame_idx"],
                "num_persons": best_frame["num_persons"],
                "num_interactions": best_frame["num_interactions"]
            },
            "social_mind_graph": graph.to_dict(),
            "mental_state_knowledge_base": mskb.to_dict(),
            "complementary_analysis": complementary,
            "qa_results": all_qa_results
        }
        
        # 保存JSON
        if output_json_path:
            with open(output_json_path, 'w', encoding='utf-8') as f:
                json.dump(full_result, f, indent=2, ensure_ascii=False, default=str)
            print(f"\n  Results saved to: {output_json_path}")
        
        # 打印最终摘要
        self._print_summary(full_result)
        
        return full_result
    
    def _process_video_collect_keyframes(self,
                                          video_path: str,
                                          output_video_path: str = None,
                                          sample_frames: List[int] = None,
                                          keyframe_interval: int = 30) -> List[dict]:
        """处理视频并收集关键帧数据"""
        
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise ValueError(f"Cannot open video: {video_path}")
        
        fps = int(cap.get(cv2.CAP_PROP_FPS))
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        print(f"  Video: {width}x{height}, {fps}FPS, {total_frames} frames")
        
        # 设置输出视频
        out_writer = None
        if output_video_path:
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            out_writer = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))
        
        # 确定要分析的帧
        if sample_frames is None:
            sample_frames = list(range(0, total_frames, keyframe_interval))
        sample_set = set(sample_frames)
        
        keyframe_data = []
        frame_count = 0
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            frame_count += 1
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_image = Image.fromarray(rgb_frame)
            
            # === Module A1: Detection + Tracking ===
            results = self.yolo(rgb_frame, verbose=False, conf=self.yolo_conf)
            
            detected_boxes = []
            detected_confs = []
            if len(results) > 0 and len(results[0].boxes) > 0:
                boxes_np = results[0].boxes.xyxy.cpu().numpy()
                confs_np = results[0].boxes.conf.cpu().numpy()
                areas = (boxes_np[:, 2] - boxes_np[:, 0]) * (boxes_np[:, 3] - boxes_np[:, 1])
                sorted_idx = np.argsort(areas)[::-1][:self.max_persons]
                for idx in sorted_idx:
                    detected_boxes.append(boxes_np[idx].tolist())
                    detected_confs.append(float(confs_np[idx]))
            
            active_persons = self.tracker.update(
                detected_boxes, detected_confs,
                frame_count, width, height
            )
            
            # === Module A2: Gaze Detection ===
            if len(active_persons) > 0:
                active_persons = self.gaze_detector.detect(pil_image, active_persons)
            
            # === Module B: Social Mind Graph ===
            self.graph.build(active_persons, width, height)
            
            # === 关键帧处理：Module C ===
            if frame_count in sample_set and len(active_persons) >= 2:
                # 运行Module C
                mskb = self.tom_engine.reason(self.graph)
                
                # 计算互动数
                num_interactions = len([
                    e for e in self.graph.edges 
                    if e.edge_type == "GAZES_AT"
                ])
                
                keyframe_data.append({
                    "frame_idx": frame_count,
                    "pil_image": pil_image.copy(),
                    "active_persons": dict(active_persons),
                    "graph": self.graph,  # 注意：graph会被下一帧覆盖
                    "graph_dict": self.graph.to_dict(),
                    "graph_nl": self.graph.to_natural_language(),
                    "mskb": mskb,
                    "mskb_dict": mskb.to_dict(),
                    "mskb_nl": mskb.to_natural_language(),
                    "num_persons": len(active_persons),
                    "num_interactions": num_interactions
                })
                
                print(f"  [Keyframe {frame_count}] "
                      f"{len(active_persons)} persons, "
                      f"{num_interactions} gaze interactions")
            
            # === 可视化输出 ===
            if out_writer and len(active_persons) > 0:
                vis_image = visualize_with_ids(
                    pil_image, active_persons, self.graph
                )
                out_writer.write(
                    cv2.cvtColor(np.array(vis_image), cv2.COLOR_RGB2BGR)
                )
            elif out_writer:
                out_writer.write(frame)
            
            if frame_count % (fps * 5) == 0:
                print(f"  Processing: {frame_count}/{total_frames} frames")
        
        cap.release()
        if out_writer:
            out_writer.release()
            print(f"  Output video saved: {output_video_path}")
        
        return keyframe_data
    
    def _select_best_keyframe(self, keyframe_data: List[dict]) -> Optional[dict]:
        """选择最具代表性的关键帧"""
        if not keyframe_data:
            return None
        
        # 评分：人数 × 2 + 互动数 × 3
        scored = [
            (kf, kf["num_persons"] * 2 + kf["num_interactions"] * 3)
            for kf in keyframe_data
        ]
        scored.sort(key=lambda x: x[1], reverse=True)
        
        best = scored[0][0]
        
        # 需要重建graph和mskb（因为graph对象会被后续帧覆盖）
        # 使用存储的数据重建
        # 注意：这里我们重新运行Module B和C以获得最新的对象
        # 实际上，我们在收集时已经保存了完整的信息
        
        # 但graph对象是共享的，需要重建
        # 最简单的方案：对最佳关键帧重新运行一次
        
        # 重新检测（确保graph和mskb是最新的）
        pil_image = best["pil_image"]
        rgb_frame = np.array(pil_image)
        
        results = self.yolo(rgb_frame, verbose=False, conf=self.yolo_conf)
        
        detected_boxes = []
        detected_confs = []
        if len(results) > 0 and len(results[0].boxes) > 0:
            boxes_np = results[0].boxes.xyxy.cpu().numpy()
            confs_np = results[0].boxes.conf.cpu().numpy()
            areas = (boxes_np[:, 2] - boxes_np[:, 0]) * (boxes_np[:, 3] - boxes_np[:, 1])
            sorted_idx = np.argsort(areas)[::-1][:self.max_persons]
            for idx in sorted_idx:
                detected_boxes.append(boxes_np[idx].tolist())
                detected_confs.append(float(confs_np[idx]))
        
        w, h = pil_image.size
        
        # 创建临时tracker来处理这一帧
        temp_tracker = PersonTracker(max_persons=self.max_persons)
        active_persons = temp_tracker.update(
            detected_boxes, detected_confs, 1, w, h
        )
        
        if len(active_persons) > 0:
            active_persons = self.gaze_detector.detect(pil_image, active_persons)
        
        fresh_graph = SocialMindGraph(fov_angle_deg=150)
        fresh_graph.build(active_persons, w, h)
        
        fresh_engine = SymbolicToMEngine(max_order=3)
        fresh_mskb = fresh_engine.reason(fresh_graph)
        
        # 更新best的graph和mskb为fresh版本
        best["graph"] = fresh_graph
        best["mskb"] = fresh_mskb
        best["graph_nl"] = fresh_graph.to_natural_language()
        best["mskb_nl"] = fresh_mskb.to_natural_language()
        
        return best
    
    def _print_summary(self, result: dict):
        """打印最终摘要"""
        print("\n" + "=" * 60)
        print("  PIPELINE EXECUTION SUMMARY")
        print("=" * 60)
        kf = result["keyframe_info"]
        print(f"\n  Keyframe: #{kf['frame_idx']} "
              f"({kf['num_persons']} persons, {kf['num_interactions']} interactions)")
        # 打印graph摘要
        graph_summary = result["social_mind_graph"].get("summary", {})
        print(f"\n  Gaze interactions: {graph_summary.get('gaze_interactions', [])}")
        print(f"  Mutual gaze: {graph_summary.get('mutual_gaze_pairs', [])}")
        # 打印MSKB统计
        mskb_data = result["mental_state_knowledge_base"]
        for order_key in sorted(mskb_data.keys()):
            entries = mskb_data[order_key]
            print(f"  {order_key}: {len(entries)} mental state entries")
        # 【修改点4】：增加准确率统计
        print(f"\n  Questions answered: {len(result['qa_results'])}")
        correct_count = 0
        for qa in result["qa_results"]:
            q_id = qa["question_id"]
            resp = qa["model_response"]
            conf = resp.get("confidence", "N/A")
            tom = resp.get("tom_order_required", "N/A")
            answer = resp.get("answer", resp.get("selected_answer_text", "N/A"))
            # 判断模型预测是否正确
            pred_idx = resp.get("selected_answer_index", -1)
            gt_answers = qa.get("ground_truth_answers", [])
            is_correct = False
            for ans in gt_answers:
                if ans.get("index") == pred_idx and ans.get("is_correct"):
                    is_correct = True
                    break
            if is_correct:
                correct_count += 1
            mark = "✓" if is_correct else "✗"
            print(f"    {mark} {q_id}: [conf={conf}, ToM={tom}] {str(answer)[:80]}...")
        total_q = len(result['qa_results'])
        if total_q > 0:
            acc = correct_count / total_q * 100
            print(f"\n  ====================================")
            print(f"  Accuracy: {correct_count}/{total_q} ({acc:.1f}%)")
            print(f"  ====================================")


# ================================================================
#               运行完整Pipeline
# ================================================================

if __name__ == "__main__":
    
    # 路径配置
    video_path = "/Users/zehao/Desktop/HAI/MultiParty/Social-IQ-2/siq2/video/Am6NHDbj6XA.mp4"
    qa_file_path = "/Users/zehao/Desktop/HAI/MultiParty/Social-IQ-2/siq2/qa/qa_train.json"
    video_output_path = "/Users/zehao/Desktop/HAI/MultiParty/code/output_full_pipeline.mp4"
    json_output_path = "/Users/zehao/Desktop/HAI/MultiParty/code/output_full_results.json"
    
    os.makedirs(os.path.dirname(video_output_path), exist_ok=True)
    
    # 初始化完整Pipeline
    pipeline = SocialMindPipeline(
        gazelle_model=gazelle_model,
        gazelle_transform=gazelle_transform,
        device=device,
        face_detector=face_detector,
        max_persons=5,
        inout_thresh=0.5,
        yolo_conf=0.7,
        max_tom_order=3
    )
    
    # 运行
    results = pipeline.process_video_and_answer(
        video_path=video_path,
        qa_file_path=qa_file_path,
        output_video_path=video_output_path,
        output_json_path=json_output_path,
        keyframe_interval=30  # 每30帧采样一次
    )
    
    # 打印完整的MSKB（最终关键帧的心智推理结果）
    print("\n\n" + "=" * 60)
    print("  FULL MENTAL STATE ANALYSIS")
    print("=" * 60)
    
    mskb_data = results.get("mental_state_knowledge_base", {})
    for order_key in sorted(mskb_data.keys()):
        print(f"\n--- {order_key} ---")
        for entry in mskb_data[order_key]:
            print(f"  [{entry['type']}] {entry['natural_language']}")
            print(f"    Rule: {entry['rule']}")
            print(f"    Confidence: {entry['confidence']}")


STEP 1: Parsing Social-IQ QA file
  Loaded 5 questions
    Am6NHDbj6XA_q5_7: Are the men comfortable with one another?...
    Am6NHDbj6XA_q1_4: How does the man in the suit feel about the man in the strip...
    Am6NHDbj6XA_q2_2: How does the man in the suit feel towards the other man at t...
    Am6NHDbj6XA_q3_3: What is the man's tone when saying he wanted to beat the keb...
    Am6NHDbj6XA_q4_0: Why does the man in the striped shirt smile at the man in th...

STEP 2: Processing video (Module A + B + C)
  Video: 640x360, 29FPS, 1799 frames
  [Keyframe 30] 2 persons, 1 gaze interactions
  [Keyframe 60] 2 persons, 2 gaze interactions
  [Keyframe 90] 2 persons, 2 gaze interactions
  [Keyframe 120] 2 persons, 2 gaze interactions
  Processing: 145/1799 frames
  [Keyframe 150] 2 persons, 2 gaze interactions
  [Keyframe 180] 2 persons, 2 gaze interactions
  [Keyframe 210] 2 persons, 1 gaze interactions
  [Keyframe 240] 2 persons, 1 gaze interactions
  [Keyframe 270] 2 persons, 1 gaze intera

In [25]:
# ================================================================
#  评估模块 + VLM Baseline 对比
#  
#  功能：
#  1. 解析Social-IQ的ground truth
#  2. 评估pipeline的QA准确率
#  3. 用VLM直接推理作为baseline对比
#  4. 生成对比报告
# ================================================================

import os
import json
import time
import base64
import re
from io import BytesIO
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass
from PIL import Image
import numpy as np
import cv2

import openai

# ================= API 配置 =================
API_KEY = "sk-bbf9f24ff4194920a43e15749a2dad29"           # ← 填写你的API Key
API_BASE = "https://dashscope.aliyuncs.com/compatible-mode/v1"  # ← 填写你的API Base URL
MODEL_NAME = "qwen-vl-plus-latest"
# 视觉模型是Qwen-VL-3.6，语言模型是qwen-plus-latest

client = openai.OpenAI(api_key=API_KEY, base_url=API_BASE)

class VideoSampler:
    """
    统一的视频帧采样器
    确保Pipeline和Baseline看到完全相同的帧
    """
    
    def __init__(self, video_path: str):
        self.video_path = video_path
        
        cap = cv2.VideoCapture(video_path)
        self.fps = int(cap.get(cv2.CAP_PROP_FPS))
        self.width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        self.duration = self.total_frames / self.fps if self.fps > 0 else 0
        cap.release()
        
        print(f"Video: {self.width}x{self.height}, {self.fps}FPS, "
              f"{self.total_frames} frames, {self.duration:.1f}s")
    
    def sample_frames(self, num_frames: int = 8, 
                       strategy: str = "uniform") -> List[dict]:
        """
        采样指定数量的帧
        """
        if strategy == "uniform":
            # 均匀分布，排除首尾各5%
            start = int(self.total_frames * 0.05)
            end = int(self.total_frames * 0.95)
            indices = np.linspace(start, end, num_frames, dtype=int).tolist()
        elif strategy == "keypoints":
            positions =[0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75, 0.85]
            indices =[int(self.total_frames * p) for p in positions[:num_frames]]
        else:
            raise ValueError(f"Unknown strategy: {strategy}")
        
        # 去重并排序
        indices = sorted(set(indices))
        return self._extract_frames(indices)
    
    def _extract_frames(self, indices: List[int]) -> List[dict]:
        """提取指定帧"""
        cap = cv2.VideoCapture(self.video_path)
        frames =[]
        
        for idx in indices:
            idx = max(0, min(idx, self.total_frames - 1))
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            
            if ret:
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                pil = Image.fromarray(rgb)
                frames.append({
                    "frame_idx": idx,
                    "pil_image": pil,
                    "timestamp": idx / self.fps if self.fps > 0 else 0
                })
        
        cap.release()
        print(f"Sampled {len(frames)} frames at indices: {[f['frame_idx'] for f in frames]}")
        return frames


# ================================================================
#  多帧图像处理工具
# ================================================================

class MultiFrameProcessor:
    """处理VLM的多图输入"""
    
    @staticmethod
    def encode_image(pil_image: Image.Image, 
                      max_size: int = 768) -> str:
        """编码为base64"""
        w, h = pil_image.size
        if max(w, h) > max_size:
            ratio = max_size / max(w, h)
            pil_image = pil_image.resize(
                (int(w * ratio), int(h * ratio)), Image.LANCZOS
            )
        buf = BytesIO()
        pil_image.save(buf, format="JPEG", quality=85)
        return base64.b64encode(buf.getvalue()).decode("utf-8")


# ================================================================
#  公平的VLM Baseline（多帧版本）
# ================================================================

class FairVLMBaseline:
    """
    极简 VLM Baseline
    输入：多帧分别作为多图独立传入
    推理：直接推理 (Direct)
    """
    
    def __init__(self, model_name: str = MODEL_NAME, 
                 max_retries: int = 3):
        self.model_name = model_name
        self.max_retries = max_retries
        self.processor = MultiFrameProcessor()
    
    def _call_api(self, messages: list, temperature: float = 0.1,
                   max_tokens: int = 2000) -> str:
        for attempt in range(self.max_retries):
            try:
                response = client.chat.completions.create(
                    model=self.model_name,
                    messages=messages,
                    temperature=temperature,
                    max_tokens=max_tokens
                )
                return response.choices[0].message.content
            except Exception as e:
                print(f"    API error (attempt {attempt+1}): {e}")
                if attempt < self.max_retries - 1:
                    time.sleep(2 ** attempt)
                else:
                    return f"[Error: {str(e)}]"
    
    def _parse_response(self, raw: str) -> dict:
        try:
            match = re.search(r'\{[\s\S]*?\}', raw)
            if match:
                result = json.loads(match.group())
                result.setdefault("selected_answer_index", -1)
                return result
        except json.JSONDecodeError:
            pass
        
        idx_match = re.search(r'(?:select|answer|index)[^\d]*(\d+)',
                               raw, re.IGNORECASE)
        if idx_match:
            return {"selected_answer_index": int(idx_match.group(1)),
                    "reasoning": raw}
        
        return {"selected_answer_index": -1, "reasoning": raw, "raw": raw}
    
    def _multi_image_input(self, frames: List[dict],
                            system_prompt: str, user_prompt: str,
                            max_images: int = 8,
                            temperature: float = 0.1) -> str:
        """将多帧作为多张图像输入"""
        if len(frames) > max_images:
            indices = np.linspace(0, len(frames)-1, max_images, dtype=int)
            frames = [frames[i] for i in indices]
        
        image_contents =[]
        for f in frames:
            b64 = self.processor.encode_image(f["pil_image"], max_size=512)
            image_contents.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{b64}"}
            })
        
        frame_desc = ", ".join([
            f"Frame {f['frame_idx']} ({f['timestamp']:.1f}s)" 
            for f in frames
        ])
        
        messages =[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content":[
                *image_contents,
                {"type": "text", "text": 
                 f"The above {len(frames)} images are sampled from a video "
                 f"at: {frame_desc}\n\n{user_prompt}"}
            ]}
        ]
        return self._call_api(messages, temperature, max_tokens=2000)
    
    def _build_prompts(self, question: str, choices: List[dict], num_frames: int) -> Tuple[str, str]:
        """构建 Direct 推理 prompt"""
        choices_text = "\n".join([
            f"  [{c['index']}] {c['text']}" for c in choices
        ])
        
        frame_context = (
            f"You are viewing {num_frames} frames sampled from a video "
            f"of a multi-party social interaction. "
            f"Consider the temporal progression across frames."
        )
        
        system_prompt = f"""You are analyzing a social interaction scene.
{frame_context}
Answer the multiple-choice question by selecting the best answer.

Respond ONLY in this JSON format:
{{"selected_answer_index": <int>, "selected_answer_text": "<text>", "reasoning": "<brief reasoning>"}}"""
        
        user_prompt = f"""Question: {question}

Answer Choices:
{choices_text}

Select the best answer by index."""

        return system_prompt, user_prompt
    
    def run_all_baselines(self, questions: List[dict],
                           frames: List[dict],
                           delay: float = 2.0) -> Dict[str, List[dict]]:
        """
        运行 baseline 矩阵，目前精简为只有一个: multi_direct
        """
        baseline_configs = [
            ("multi_direct", "multi_image", "direct"),
        ]
        
        results = {name:[] for name, _, _ in baseline_configs}
        
        for q_idx, q in enumerate(questions):
            print(f"\n  [{q['question_id']}] {q['question'][:55]}...")
            
            for name, input_mode, reasoning_mode in baseline_configs:
                print(f"    {name}...", end=" ", flush=True)
                
                # 直接获取 prompt 并调用 multi_image_input
                sys_prompt, user_prompt = self._build_prompts(
                    q["question"], q["answers"], len(frames)
                )
                
                raw_ans = self._multi_image_input(frames, sys_prompt, user_prompt)
                ans = self._parse_response(raw_ans)
                ans["method"] = name
                ans["num_frames"] = len(frames)
                
                results[name].append(ans)
                
                sel_idx = ans.get("selected_answer_index", -1)
                if 0 <= sel_idx < len(q["answers"]):
                    # label = q["answers"][sel_idx]["label"]
                    is_correct = q["answers"][sel_idx].get("is_correct", False)

                    mark = "✓" if is_correct == "a" else "✗"
                else:
                    mark = "?"
                print(f"{mark}")
                
                time.sleep(delay)
        
        return results


# ================================================================
#  评估器
# ================================================================

class SocialIQEvaluator:
    
    @staticmethod
    def parse_qa_file(filepath: str, target_vid_name: str = None) -> List[dict]:
        """解析 Social-IQ 2.0 的 JSON/JSONL QA文件"""
        questions = []
        
        with open(filepath, 'r', encoding='utf-8') as f:
            content = f.read().strip()
            
            # 兼容处理：检查是标准JSON列表 还是 JSON Lines
            if content.startswith('['):
                data_list = json.loads(content)
            else:
                data_list = [json.loads(line) for line in content.split('\n') if line.strip()]
        
        for data in data_list:
            # 如果指定了 vid_name，过滤掉不属于该视频的QA
            if target_vid_name and data.get("vid_name") != target_vid_name:
                continue
            
            q_id = data.get("qid")
            q_text = data.get("q")
            correct_idx = data.get("answer_idx", -1)
            
            current_q = {
                "question_id": q_id,
                "question": q_text,
                "vid_name": data.get("vid_name"),
                "ans_corr": data.get("ans_corr", ""),  # 保存正确答案文本供参考
                "answers": [],
                "correct_indices": [],
                "incorrect_indices": []
            }
            
            # Social-IQ 2 默认有4个选项 a0, a1, a2, a3
            for i in range(4):
                ans_key = f"a{i}"
                if ans_key in data:
                    text = data[ans_key]
                    is_correct = (i == correct_idx)
                    label = 'a' if is_correct else 'i'  # 兼容旧代码的 label
                    
                    current_q["answers"].append({
                        "index": i, 
                        "label": label,
                        "text": text, 
                        "is_correct": is_correct
                    })
                    
                    if is_correct:
                        current_q["correct_indices"].append(i)
                    else:
                        current_q["incorrect_indices"].append(i)
            
            questions.append(current_q)
            
        return questions
    
    @staticmethod
    def evaluate_batch(questions: List[dict], 
                       model_answers: List[dict]) -> dict:
        results =[]
        correct = 0
        
        for q, ans in zip(questions, model_answers):
            sel_idx = ans.get("selected_answer_index", -1)
            if sel_idx is None:
                sel_idx = -1
            sel_idx = int(sel_idx)
            
            if 0 <= sel_idx < len(q["answers"]):
                is_correct = q["answers"][sel_idx]["is_correct"]
                sel_label = q["answers"][sel_idx]["label"]
                sel_text = q["answers"][sel_idx]["text"]
            else:
                is_correct = False
                sel_label = "invalid"
                sel_text = "N/A"
            
            results.append({
                "question_id": q["question_id"],
                "correct": is_correct,
                "selected_label": sel_label,
                "selected_text": sel_text
            })
            if is_correct:
                correct += 1
        
        total = len(questions)
        return {
            "accuracy": correct / total if total > 0 else 0,
            "correct": correct,
            "total": total,
            "per_question": results
        }


# ================================================================
#  完整的公平对比实验
# ================================================================

class FairComparisonExperiment:
    """
    公平对比实验
    保证：Pipeline 和 Baseline 看到完全相同的帧
    """
    
    def __init__(self):
        self.evaluator = SocialIQEvaluator()
        self.vlm = FairVLMBaseline(model_name=MODEL_NAME)
    
    def run(self, 
            video_path: str,
            qa_file_path: str,
            pipeline_result_path: str = None,
            output_path: str = None,
            num_sample_frames: int = 8,
            api_delay: float = 2.0) -> dict:
        
        print("=" * 70)
        print("  FAIR COMPARISON EXPERIMENT")
        print("  Pipeline and all baselines see the SAME frames")
        print("=" * 70)
        
        # ===== 1. 解析QA =====
        # questions = self.evaluator.parse_qa_file(qa_file_path)
        # print(f"\nLoaded {len(questions)} questions")

        vid_name = os.path.splitext(os.path.basename(video_path))[0]
        questions = self.evaluator.parse_qa_file(qa_file_path, target_vid_name=vid_name)
        print(f"\nLoaded {len(questions)} questions for video: {vid_name}")
        
        # ===== 2. 统一采样帧 =====
        print(f"\n--- Sampling {num_sample_frames} frames ---")
        sampler = VideoSampler(video_path)
        
        pipeline_results = None
        if pipeline_result_path and os.path.exists(pipeline_result_path):
            with open(pipeline_result_path, 'r') as f:
                pipeline_results = json.load(f)
        
        # 均匀采样（这些帧Pipeline和Baseline都会看到）
        frames = sampler.sample_frames(
            num_frames=num_sample_frames, 
            strategy="uniform"
        )
        print(f"  All baselines will see frames: {[f['frame_idx'] for f in frames]}")
        
        # ===== 3. 收集Pipeline结果 =====
        print("\n--- Pipeline Results ---")
        pipeline_answers =[]
        if pipeline_results and "qa_results" in pipeline_results:
            for qa_r in pipeline_results["qa_results"]:
                pipeline_answers.append(qa_r["model_response"])
            print(f"  Loaded {len(pipeline_answers)} pipeline answers")
        else:
            print("  No pipeline results available")
            pipeline_answers = [{"selected_answer_index": -1}] * len(questions)
        
        # ===== 4. 运行VLM Baseline =====
        print("\n--- Running VLM Baseline ---")
        print(f"  Baseline sees {len(frames)} frames directly")
        
        baseline_results = self.vlm.run_all_baselines(
            questions, frames, delay=api_delay
        )
        
        # ===== 5. 评估所有方法 =====
        print("\n" + "=" * 70)
        print("  EVALUATION")
        print("=" * 70)
        
        all_evals = {}
        
        # Pipeline
        ev_pipeline = self.evaluator.evaluate_batch(questions, pipeline_answers)
        all_evals["Ours (Neuro-Symbolic)"] = ev_pipeline
        
        # Baselines
        for method_name, answers in baseline_results.items():
            ev = self.evaluator.evaluate_batch(questions, answers)
            all_evals[method_name] = ev
        
        # ===== 6. 生成报告 =====
        report = self._generate_report(all_evals, questions, len(frames))
        print(report)
        
        # ===== 7. 保存 =====
        output = {
            "experiment_config": {
                "video": video_path,
                "qa_file": qa_file_path,
                "num_frames_sampled": len(frames),
                "frame_indices": [f["frame_idx"] for f in frames],
                "model": MODEL_NAME,
                "fairness": "All methods see identical frames"
            },
            "evaluations": {
                method: {
                    "accuracy": ev["accuracy"],
                    "correct": ev["correct"],
                    "total": ev["total"],
                    "per_question": ev["per_question"]
                }
                for method, ev in all_evals.items()
            },
            "raw_responses": {
                method:[
                    {"question_id": questions[i]["question_id"],
                     "response": ans}
                    for i, ans in enumerate(answers)
                ]
                for method, answers in baseline_results.items()
            },
            "report": report
        }
        
        if output_path:
            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(output, f, indent=2, ensure_ascii=False, default=str)
            print(f"\nResults saved to: {output_path}")
        
        return output
    
    def _generate_report(self, all_evals: dict, 
                          questions: list,
                          num_frames: int) -> str:
        """生成对比报告"""
        lines =[]
        lines.append("\n" + "=" * 70)
        lines.append("  FAIR COMPARISON REPORT")
        lines.append(f"  All methods see the same {num_frames} frames")
        lines.append("=" * 70)
        
        # ===== 分组显示 =====
        groups = {
            "Our Method":[],
            "Multi-Image (N帧→N图)":[]
        }
        
        for method, ev in all_evals.items():
            if "Ours" in method or "Neuro" in method:
                groups["Our Method"].append((method, ev))
            elif "multi" in method:
                groups["Multi-Image (N帧→N图)"].append((method, ev))
        
        for group_name, methods in groups.items():
            if not methods:
                continue
            
            lines.append(f"\n  ┌─ {group_name} {'─' * (50 - len(group_name))}┐")
            for method, ev in methods:
                acc = ev["accuracy"]
                c = ev["correct"]
                t = ev["total"]
                bar = "█" * int(acc * 20) + "░" * (20 - int(acc * 20))
                
                short_name = method.replace("Ours (Neuro-Symbolic)", "★ Ours")
                short_name = short_name.replace("multi_", "m_")
                
                lines.append(f"  │ {short_name:<25} {bar} {acc:>5.1%} ({c}/{t}) │")
            lines.append(f"  └{'─' * 58}┘")
        
        # ===== 逐题对比表 =====
        lines.append("\n  --- Per-Question Results ---\n")
        
        show_methods =["Ours (Neuro-Symbolic)", "multi_direct"]
        show_methods =[m for m in show_methods if m in all_evals]
        
        # 表头
        header = f"  {'Q':<5}"
        for m in show_methods:
            short = m.replace("Ours (Neuro-Symbolic)", "Ours")
            short = short.replace("multi_", "m.")
            header += f" {short:>8}"
        lines.append(header)
        lines.append("  " + "─" * (5 + 9 * len(show_methods)))
        
        for q in questions:
            row = f"  {q['question_id']:<5}"
            for m in show_methods:
                ev = all_evals[m]
                pq = next(
                    (p for p in ev["per_question"] 
                     if p["question_id"] == q["question_id"]), None
                )
                if pq and pq["correct"]:
                    row += f"    {'✓':>4}"
                elif pq:
                    row += f"    {'✗':>4}"
                else:
                    row += f"    {'?':>4}"
            lines.append(row)
        
        # ===== 关键分析 =====
        lines.append("\n  --- Key Findings ---\n")
        
        ours_key = "Ours (Neuro-Symbolic)"
        baseline_key = "multi_direct"
        
        if ours_key in all_evals and baseline_key in all_evals:
            ours_acc = all_evals[ours_key]["accuracy"]
            baseline_acc = all_evals[baseline_key]["accuracy"]
            delta = ours_acc - baseline_acc
            
            lines.append(f"  vs Baseline ({baseline_key}):")
            lines.append(f"    Ours: {ours_acc:.1%} vs Baseline: {baseline_acc:.1%} (Δ = {delta:+.1%})")
            
            if delta > 0:
                lines.append(f"    → ★ Our neuro-symbolic framework adds {delta:+.1%} over")
                lines.append(f"      the baseline WITH SAME VISUAL INPUT.")
                lines.append(f"      This improvement comes purely from structured reasoning.")
            elif delta == 0:
                lines.append(f"    → Tied with baseline.")
            else:
                lines.append(f"    → ⚠ Underperforms baseline. Investigate errors.")
            
            # 找到只有我们答对的题
            ours_correct = {p["question_id"] for p in all_evals[ours_key]["per_question"] if p["correct"]}
            baseline_correct = {p["question_id"] for p in all_evals[baseline_key]["per_question"] if p["correct"]}
            
            unique_ours = ours_correct - baseline_correct
            if unique_ours:
                lines.append(f"\n  ★ Questions ONLY our method answered correctly: {unique_ours}")
                for q_id in unique_ours:
                    q_data = next(q for q in questions if q["question_id"] == q_id)
                    lines.append(f"    {q_id}: {q_data['question'][:65]}...")
        
        # ===== 总结表 =====
        lines.append("\n  " + "=" * 50)
        lines.append("  SUMMARY TABLE (for paper)")
        lines.append("  " + "=" * 50)
        lines.append(f"\n  {'Method':<30} {'Accuracy':>10} {'Frames':>8}")
        lines.append("  " + "─" * 50)
        
        # 按准确率排序
        sorted_evals = sorted(all_evals.items(), 
                               key=lambda x: x[1]["accuracy"], reverse=True)
        for method, ev in sorted_evals:
            if "Ours" in method:
                method_display = f"★ {method}"
            else:
                method_display = f"  {method}"
            lines.append(f"  {method_display:<30} {ev['accuracy']:>9.1%} {'N':>8}")
        
        return "\n".join(lines)


# ================================================================
#  运行入口
# ================================================================

if __name__ == "__main__":
    
    # 路径配置
    video_path = "/Users/zehao/Desktop/HAI/MultiParty/Social-IQ-2/siq2/video/Am6NHDbj6XA.mp4"
    qa_file_path = "/Users/zehao/Desktop/HAI/MultiParty/Social-IQ-2/siq2/qa/qa_train.json"
    # qa_file_path = "/Users/zehao/Desktop/HAI/MultiParty/Social-IQ/raw/qa/_0at8kXKWSw_trimmed.txt"
    pipeline_result_path = "/Users/zehao/Desktop/HAI/MultiParty/code/output_full_results.json"
    comparison_output_path = "/Users/zehao/Desktop/HAI/MultiParty/code/fair_comparison_results.json"
    
    os.makedirs(os.path.dirname(comparison_output_path), exist_ok=True)
    
    # 运行公平对比
    experiment = FairComparisonExperiment()
    results = experiment.run(
        video_path=video_path,
        qa_file_path=qa_file_path,
        pipeline_result_path=pipeline_result_path,
        output_path=comparison_output_path,
        num_sample_frames=50    # 采样8帧
        # api_delay=2.0           # API调用间隔
    )

  FAIR COMPARISON EXPERIMENT
  Pipeline and all baselines see the SAME frames

Loaded 5 questions for video: Am6NHDbj6XA

--- Sampling 50 frames ---
Video: 640x360, 29FPS, 1799 frames, 62.0s
Sampled 50 frames at indices: [89, 122, 155, 188, 221, 254, 287, 320, 353, 386, 419, 452, 485, 518, 551, 584, 617, 651, 684, 717, 750, 783, 816, 849, 882, 915, 948, 981, 1014, 1047, 1080, 1113, 1146, 1180, 1213, 1246, 1279, 1312, 1345, 1378, 1411, 1444, 1477, 1510, 1543, 1576, 1609, 1642, 1675, 1709]
  All baselines will see frames: [89, 122, 155, 188, 221, 254, 287, 320, 353, 386, 419, 452, 485, 518, 551, 584, 617, 651, 684, 717, 750, 783, 816, 849, 882, 915, 948, 981, 1014, 1047, 1080, 1113, 1146, 1180, 1213, 1246, 1279, 1312, 1345, 1378, 1411, 1444, 1477, 1510, 1543, 1576, 1609, 1642, 1675, 1709]

--- Pipeline Results ---
  Loaded 5 pipeline answers

--- Running VLM Baseline ---
  Baseline sees 50 frames directly

  [Am6NHDbj6XA_q5_7] Are the men comfortable with one another?...
    multi_direct